<div dir="rtl">

# HM_07 - מעבדה: שירות עם חוזה - API, מסד נתונים, תור ו-Docker (בקטן)

**הקורס:** מודלי שפה וסוכנים ממוחשבים למסלול מערכות מידע (4261310) ולמסלול ידע ונתונים (תשפז_ב_4261110) · מאת ד״ר ולד לנדא וד״ר רן דובין · **מפגש 7:** APIs, מסדי נתונים ו-Backend

**שם:** _(כתבו כאן)_

---

## מה עושים במעבדה

במפגש 7 למדנו איך הופכים קוד שעובד ב-Notebook ל**שירות**: HTTP ותגובות, **OpenAPI כחוזה**, שירות FastAPI, מסד נתונים עם סכמה ו-SQL, תור ועובד, ו-Docker. הכלל של המפגש:

> **החוזה קודם: מגדירים מה נכנס, מה יוצא ומה נחשב נכון - ואז בודקים. ושירות איטי שמים מאחורי תור.**

במעבדה הזו לא תפעילו חבילת שירות על המחשב: **הכול רץ כאן ב-Colab**, ותכתבו בעצמכם את החלקים הקטנים שמרכיבים שירות ושבודקים אותו - פונקציה בצד השרת, **כלי שבודק שרת מול החוזה שלו**, שאילתות SQL, ו**בדיקה מקצה לקצה**.

**כנות לגבי מה שרץ כאן:** ב-Colab אין Docker, Kubernetes, Streamlit ו-Redis. לכן:
- שרת **FastAPI אמיתי** רץ בתוך המחברת (ברקע, על `127.0.0.1`), ואתם פונים אליו עם `requests`. אלה תגובות HTTP אמיתיות.
- מסד **SQLite אמיתי** (בזיכרון) עם נתוני החנות, שאילתות SQL אמיתיות.
- **סימולציה** קטנה של תור ועובד (`queue.Queue` ו-threads): אותו מבנה כמו Celery + Redis + Flower, אבל בלי להתקין אותם. התור האמיתי (Redis, Celery, Flower) רץ במעבדות של המפגש ובפרויקט, על המחשב שלכם.
- קבצי **Dockerfile ו-compose** נקראים ונבדקים כקוד (בלי להריץ קונטיינר).

| חלק | נושא | מה מממשים | זמן |
|---|---|---|---|
| **בכיתה (חובה)** | | | |
| א' | HTTP ו-OpenAPI | `status_family`, `find_product`, `create_order`, `list_endpoints`, `check_contract` | 13 דק' |
| ב' | SQL וסכמה | `q_slow_suppliers`, `q_category_revenue`, `q_top_products`, `join_multiplies`, `find_products_safe` | 12 דק' |
| ג' | משלבים: API ↔ מסד ↔ AI | `top_products_from_db`, `e2e_test`, endpoint שכותב Gemini ובדיקתו | 10 דק' |
| ד' | מסקנה | - | 2 דק' |
| **בהמשך / בית** | | | |
| ה' | תור ו-worker (Celery בקטן) | `submit_job`, `worker_loop`, `get_status`, `status_counts` | 12 דק' |
| ו' | Docker ו-Kubernetes (קריאה והבנה) | `parse_dockerfile`, כללי lint, `compose_services`, `k8s_replicas` | 12 דק' |
| ז' | קריאה ל-API חיצוני (אופציונלי) | `get_rate` | 6 דק' |

**חובה בכיתה: א'+ב'+ג'+ד' (37 דק').** חלקים ה' ו-ו' הם חלק מהסילבוס ומומלץ מאוד להשלים בבית; ז' אופציונלי.

## איך עובדים

1. **מממשים** כל פונקציה שמסומנת `# TODO` - **בעצמכם, בלי כלי AI**. המטרה היא להבין מה השירות עושה ואיך בודקים אותו, לא לקבל קוד. (רק בחלק ג' יש צעד שבו משתמשים בכלי AI, כי הוא נושא הניסוי.)
2. **מריצים את תא הבדיקה** שאחרי כל פונקציה. כשהמימוש נכון יודפס `✓`; אחרת תופיע הודעה שמסבירה מה נכשל.
3. **לפני כל הרצה מפתיעה כותבים תחזית** בתא המיועד לכך. תחזית "שגויה" עם ניתוח טוב שווה יותר מתחזית "נכונה" בלי הסבר.
4. **עונים על שאלות הניתוח** בתאי ה"תשובה", במונחים של מה שלמדנו בהרצאה (חוזה, קודי סטטוס, שכבות, תור).

> **אל תיבהלו מהקוד.** כל פונקציה כאן היא 2-8 שורות, ובכל שלב מופיעים הסבר, דוגמה ורמז. אם נתקעתם: קראו שוב את ה-docstring (הטקסט בין שלוש המרכאות), והסתכלו על הודעת הבדיקה - היא מכוונת. ואם הבדיקה נכשלת בגלל שהשרת "קורס" (500) - קראו את ההודעה, לא רק את הקוד.

## הגשה

בסוף: **Runtime ← Restart session and run all**, ודאו שתא הבדיקה המסכם בסוף המחברת מראה `✓` לכל הפונקציות שהגעתם אליהן, ואז **File ← Download ← Download .ipynb** והגישו את הקובץ.

</div>

In [ ]:
# תא הכנה 1 - התקנה (ב-Colab החבילות fastapi ו-uvicorn לא תמיד מותקנות).
# אם Colab מבקש "Restart session" אחרי ההתקנה - אשרו, והריצו מההתחלה (Runtime ← Run all).
try:
    import fastapi, uvicorn, yaml
except ImportError:
    %pip install -q fastapi uvicorn pyyaml
print("✓ החבילות מוכנות")

In [ ]:
# תא הכנה 2 - ייבוא
import re, json, time, queue, threading, itertools, socket, sqlite3, inspect
from collections import Counter
from typing import Optional

import requests
import pandas as pd
import yaml
import uvicorn
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

print("✓ מוכן")

In [ ]:
#@title תא הכנה - הנתונים (אפשר לא לפתוח) { display-mode: "form" }
import base64, gzip, pathlib
_FILES = {
    'products.csv': (
        'H4sIAPkR2WoC/21Z227cyBF9D5B/mA9gCPa9+3Etx+vE66xgOTDytKBHtDTQaDiYSxzn6/dUNfvG8ZMk6DR5qvrUlcfT/HjdXv7Y'
        'PXaH8XXqtuNleppPP7rrYXf5YzufL/G342m3nbrz9Xjc76YT4H/9y/0wiO7tdH7Z/Da+HjdvxvNu2/3+7RsBZR98Z3ulu4fBMFRW'
        '0Pv99ZyQVvdBdUKYXhuAPYNVDT7NCat971wXXD+U5+oK+nF32CWssr3vjOxNANQy1HT/mi/T13l+2fxiWr4u9E6Aheo94SXjbYOv'
        'SYOIBXyAmZmIa9GFtRQE1ro3hbVvwDVv03vThV7JTDt0v077zf102DxMl5a2kuQK63rnk+/E0MAbV7s+hE5I2UuR4aKFF9pK9zp0'
        'BlToXhyDZQNuaNve6c6L3hATwWjVPVzG4346taSNIuu86oXMUJ2hNWHnew03K9MPIt2KMAVayOL1QfKF8EMVI21GNsKAh0Xnht7m'
        'R7qood9PT+Nh9/8bvkPvbOdDH1z2ml+faGgPPXERvTPpwkW4OVCRH3qEB3zHVjJeDmt8bQLJjyRbIgYX+uV5R4oaT4+bj+PpZW2F'
        'lb31UeFkuOZT8ienakuk6g2ku8QmX5VUPztTjBGOZCNtP2RJSv2TI7U9zvWGqVmfVC/pmnfblx8bCpTzyhbbWxIyPFBeYtsDTbgq'
        'ujvre6My3K3gxQKEibKgo3sCLVb7Fl6zh2cFZTBHYl0kJUN3Px5h5t1+dwT7+X+tBXqgcEHuKxJRw82Rxgbbi8XwJRqVuD3QWIE3'
        'COGrjIawW59YGeJCF5DURLoGpbqPM/L/fNoglg6PrRUC6cd1eEXIKQIZvz1Q2wAXKWjDUD5MlMwaX2UgSt4QkspguwI3QaEoc3pH'
        'wZrwbvnn5u553EFz4yp/xrSB0JYyX4O/PVPbYHyPYA4mJjpONLDp9khVtSgnLRfH2tPDLb65BxZgQPJTKVKhrH8/vPnb3eZu/Lqf'
        'FiM+T9vnTsFLnUWW9skCLRssk2eoGDgF4OkiUccbGixYM9SZXpGmB6rj6bm6wTJjBkPMKG1Ox5uKzzXdl91p2k9nxMp8PbeUuV7h'
        'FTbXCm3X+ELbKS62VI98ulntbvCJuiX5Qpk+5xIcW4EL94Dci2BXopcqkwndh+lHzFY17dAbR2aqXPTNUJCFMPLlQKHKNZxvHEm0'
        '4LKPETeIUFwglb/oNyMLsJCk5khTTrUyxRky2Zfp63Z8rRmikBiuVCYLBxez4Bp+qgss/fQ0k1HFjfAx7r/IHI9eQIWZ4wQgyYeL'
        '8+Ci++f5MDX5Ir5WklJtbIMiOd9gK4bch6GO+nwpSAUNNvFE1cArUaiy9uzQICs3InQB5QK1GA5h3c/fuVYeXmqyUvdeMuWcTeD8'
        'ClvISs+NkouNUqSgGmjiCpYODxXMNb5f18BCFT0aWgLqGXOJt6Z7//bjPza/PI7HS67uUZkwDK415IlFmda26CqcbCxvhn4sF2Hd'
        'Cp0YK35wb21m4VtkoWwEaQHVw+QODCnvN8Dm4+ZhP03/bZKAYyBFdRaYG1bwQhpBgkYXnaPIyka/vkJXkoDzJEpsxsoVtkpe6KVQ'
        'NmT0RuShujf763SZ58vz5uE4jS+tuxEVaJtF4A6SrxGV9vZElXgdpVG51EnOSaj7PzmRLCBfKm5nc8ZDVbs9UMyABGCAq+cXjEp/'
        'H09fr4/nRtiKijynqEUqyMUJV1H2PXoeZHaVKyOOZVxWtKPpzYqoU36tHzKsCr1Ab0XiZ3FEnOju5m/fJlzG9Wlh+GF32T5PBy6y'
        'lga9PLwhGCs080xgcEBFxj3yVMPJAmmjRoNtAqMJxPVS68jZj9XhdY1m0gmuXY8Q9aYMC+gxvowcf/PlkktxfrxB6x94zlq8hm67'
        'wTfUFdMga1U21K3wFXlIVZG6XR6cUOQadEMeV013qGi0TewRkdfD9rnqSDMaCcGRMFxITw9DhW6Ia+55HcdOAosaXLHWgcMFYnaJ'
        'Bowu2IYzVRxJI7PPoQKB3V0vl90BKqlqctZKoLKIKyo+R1ltTzTcDU1AIY5B8QVmDa/YU39lSeE297jBrvCNBYAi+wQd/cgJBcPH'
        '3fP0bfPhsPu2Vgx0i+YHnnTl+b6GN9z9wNsS3BXNwVG+sLuG156nESBEcVH2FsNQQ1ulS2qcMQIMC2sMsIjdcfNh+onOMXwq6slk'
        'bDUiXtb4Vi6BagPIpwFXIL3V4Iq05EEbGdCmCxJI/RW4ob30UKWpEIPp3s3z4+ZuPlzG3SGn7iqkkUzQMQ8mM7frI63TPXXwQmGs'
        'T3ckYM76SGWD0by4GAbaVyVefn1i7X+kPC/iO+IJCOhIY8KncfuyskLxdgHzS17loFmv4a3/DU04uLS8ghIIyRpdkad5TtELRKrM'
        'gpYqBdzwphWUolaNtyN8XdDb2935+We00fQ7z80Hr/DYleh+C7yhbTm/4O2xC5WMNjW61o0jIhiTB5dttBW2TTOGKpbUsRUUDMax'
        'cT8ivczf9+scI2gcci5OIPHZvoa3CcZSLkJL4VS2MTToVir4J+JfprZJyKEGt8lF0yqGNmIqxx0U8Pn5NH/f3O/2e/yI1N/PrxhW'
        'DSUvxA81vgtxvKeBM3VGQ+BUSF2f+kKBmajFgjhDhabFkOQdQ3qubrHMOz7X0SIYzZvPwSCpiu73m7v9nDXCWFoLwj7bp8WOkLaG'
        'FrZOpMnN2+wK12ATW5g0xAKUioqQvkZWXHnlSNehk+Jol7PbXq6nafPuNL5OjYPRIJGoh9R6CFrjNOjC2JjYeXtKQIvXaInTwrOL'
        'Ha3Q0B+H7DXa3zTgwhuCphZTxhaTHUcKucyn8YkZv+TdccQrHimHOMyynbSsb/GVNPQy4ww5EGnaWcETdcgTXZORZc8jqLtu0YU7'
        'BiLB1UHn60EncofhbT+t1Iysge4q7/SF8glXuApBt+3iTBpfHjIqUXQqrkN1DCS+Cz0kWOFmOHRQIajOLC+FQN7Oc71OSjJHtreq'
        'yoRaFmQhSJbi3YiePN0JrSpkIolSiCJohyiviNMFV1jSuBjI7dmB5MpnnizvrieqN40jHW1HUMPiBoYzFJqQ1YHKo/wRABooxV67'
        'G3i+fUUitCo2QHwBlDZbdBVxkrtUHxc20cjQvR8PT4A/tKpFwkZK0qK6NDPU2MIZLRK6TNxJjmMkrBqZnawpL+GZaSkg0IBWwIop'
        'x6JZ1ivsZvx2vx8PeNzcEEXBgEODjUTZKJTeAq3EwN+FREqlHOeQRQXNfvUkG3RoPq2lBZJ7ARai6CK85iHHFqjr3owYHT/jGvaN'
        'Ty3tn2B/7lcFqm2FrbIAt7bxe0jyVWigObws1QfaidssXTvU0Eq8vN+jEhuSA/D3f+ansYqwh+N8upyptIJsQJdkkwDQ/GcsU12g'
        '0JWgxtO5TEBVSDBdgPR5IybxdKtIXhnIPBckhhPaitrykUugP/nn9fW4+TQfp5aqdbFFdvESol22Qtdkre01t9PKpUvAqQpb0Y2f'
        'J4YhNnN8Cwjjgq0Zk1oCf4tS+XZt6N5eX79+nVD+zMtTy5p2xFQWdE4MbmjhNW1uuZBvQ+4YMGq26EI8KKp7gmSc0xQm2QZec4/5'
        '0cjYKvINoqH6NJ1358t42JK784IxHbFUBfVQ9vkCUbU+0jie+m1ac6eRWqBHuDlQjFA8q6Pd1VnXCIb1gUY0vGjF1drcDZIk+Z/t'
        'EqHYjehFI5snQkGfd5sDzSVY+qAp6DNzzooUy+2BYoLnXRoJU+cJyw8rfG2B1hT26LdMwYvu1x+0e17JByQwWNCklGcH3HmCNqxj'
        'J4saVMz0qkALX8ujYBhir84qg38SsCYa+4OYH+MDcZfXw4En9Wm/yiVaUdGkzxY+P9e2+Jqw8VHsXGqXq8ebWnxhTSOOo+KSv+QL'
        'FLgGXVNXPMIaXVUBBO67eXxFUO/LN/TydENqL+kiDA26YT7whQ+qbGtEEC28EKfvMvwxVqdPkwL6rcE1b4iP0HX1CDTaw0xaO+73'
        '5xVxThlwi805IOgWX1MXnLikiuuOyNys4JVSuDpQs1b6GqSmBt7oRdLqCNWkRD+tasbjTVahlSlSRW5oaEUzHhuqOi6KfK+zI0IE'
        'NVI29OF6+ZJDb5S0kxmPrRiodNGGYulz/wSL9kifpSMAAA=='
    ),
    'suppliers.csv': (
        'H4sIAPkR2WoC/03MTQqDMBCG4b3gHXqAQRITrVm2atu9BwiBzCLgH/4svL2MBqbLeXjnW/d57gMuNngY3YDQo/N2CwNa744VpvE+'
        'FrdhmnRCQhc/Hi9QILKKNGd9g9TEhlyx11dtCmLN3MT8minYW5CC+py8ZP/cvdHkT/bvf1+x/+K+SpMTbHGZzusAAAA='
    ),
    'orders.csv': (
        '...'
    ),
}
for _name, _b64 in _FILES.items():
    pathlib.Path(_name).parent.mkdir(parents=True, exist_ok=True)
    pathlib.Path(_name).write_bytes(gzip.decompress(base64.b64decode(_b64)))
print('✓ הקבצים נוצרו:', ', '.join(_FILES))

In [ ]:
#@title תא הכנה - פונקציות עזר: הפעלת שרת והדפסת תגובה (אפשר לא לפתוח) { display-mode: "form" }
import os
os.environ["NO_PROXY"] = "127.0.0.1,localhost"      # פניות ל-localhost לא עוברות דרך פרוקסי

def start_server(app):
    """מפעיל את אפליקציית ה-FastAPI עם uvicorn בתהליכון (thread) ברקע. מחזיר (server, base_url)."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        port = s.getsockname()[1]                       # פורט פנוי
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="critical")
    server = uvicorn.Server(config)
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)
    else:
        raise RuntimeError("השרת לא עלה")
    return server, f"http://127.0.0.1:{port}"

def show(method, path, base=None, **kwargs):
    """שולח בקשת HTTP ומדפיס: שורת סטטוס, סוג התוכן וגוף התגובה (מקוצר)."""
    r = requests.request(method, (base or BASE) + path, timeout=10, **kwargs)
    body = r.text if len(r.text) < 170 else r.text[:167] + "..."
    print(f"{method} {r.url.replace(base or BASE, '')}  →  {r.status_code} {r.reason}")
    print(f"    content-type: {r.headers.get('content-type')}")
    print(f"    body: {body}")

print("✓ פונקציות העזר מוכנות")

<div dir="rtl">

---

# חלק א' - HTTP ו-OpenAPI: מדברים עם שירות ובודקים אותו לפי החוזה (13 דק')

## הרקע

כל תקשורת בין לקוח לשרת היא **HTTP**: הלקוח שולח **בקשה** (שיטה + כתובת), והשרת מחזיר **תגובה** = *שורת סטטוס* (קוד ומשמעות) + *כותרות* + *גוף* (בדרך כלל JSON). **קוד הסטטוס** הוא מה שמאפשר לתוכנה לדעת אם הצליח, בלי לפענח טקסט. **הספרה הראשונה אומרת הכול:**

| קוד | משמעות | דוגמה |
|---|---|---|
| 200 | הצליח | `GET /products` |
| 201 | נוצר | `POST /orders` |
| 202 | התקבל, יטופל ברקע | `POST /jobs` (חלק ה') |
| 404 | לא נמצא | `GET /products/NOPE` |
| 422 | הקלט לא חוקי (לפי החוזה) | `GET /products?limit=0` |
| 429 | יותר מדי בקשות | API חיצוני (חלק ז') |
| 500 | תקלה בשרת | באג בקוד השירות |

**2xx** = הצליח · **4xx** = הלקוח טעה (אל תנסו שוב אותה בקשה) · **5xx** = השרת נכשל (לא אשמת הבקשה).

**OpenAPI** הוא **החוזה**: קובץ (`openapi.json`) שמתאר אילו כתובות קיימות, מה אפשר לשלוח, ואילו תגובות אפשריות. **FastAPI מייצר אותו מהקוד**, ו-**Swagger UI** (`/docs`) הוא רק ממשק שמציג אותו. מהחוזה אפשר לגזור **בדיקות** - וזה מה שנבנה כאן.

## שלב 0 - שירות אמיתי בתוך ה-Colab

הקוד למטה מגדיר שירות קטן של החנות (FastAPI), ו-`uvicorn` מפעיל אותו ברקע בכתובת `127.0.0.1`. זה שרת אמיתי: מי שפונה אליו עם `requests` מקבל תגובות HTTP אמיתיות. **קראו את הקוד (אין צורך לשנות):**

| בקוד | מה זה |
|---|---|
| `@app.get("/products/{product_id}")` | שיטה וכתובת; `product_id` מגיע מהכתובת |
| `Query(20, ge=1, le=200)` | **גבולות** - מוגדרים פעם אחת, ונכנסים לחוזה; מחוץ להם FastAPI מחזיר 422 |
| `class OrderIn` | מבנה גוף הבקשה (`quantity` חייב להיות גדול מ-0) |
| `status_code=201`, `responses={404: ...}` | אילו קודים *מתועדים* בחוזה |
| `find_product`, `create_order` | **שתי פונקציות שעוד לא כתבתם** - בשלבים 2 ו-3 |

</div>

In [ ]:
PRODUCTS = {
    row.product_id: {"product_id": row.product_id, "name": row.name, "category": row.category,
                     "unit_price": row.unit_price, "supplier_id": row.supplier_id}
    for row in pd.read_csv("products.csv").itertuples()
}
ORDERS = {}      # הזמנות שנוצרו דרך ה-API: order_id -> הזמנה


class OrderIn(BaseModel):
    """גוף הבקשה של POST /orders. quantity חייב להיות > 0 - אחרת FastAPI מחזיר 422 בעצמו."""
    product_id: str
    quantity: int = Field(gt=0)


app = FastAPI(title="Store API", version="1.0.0")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/products")
def list_products(category: Optional[str] = None, limit: int = Query(20, ge=1, le=200)):
    items = [p for p in PRODUCTS.values() if category is None or p["category"] == category]
    return items[:limit]


@app.get("/products/{product_id}", responses={404: {"description": "Product not found"}})
def get_product(product_id: str):
    return find_product(product_id)            # את find_product תכתבו בשלב 2


@app.post("/orders", status_code=201, responses={404: {"description": "Product not found"}})
def post_order(order: OrderIn):
    return create_order(order)                 # את create_order תכתבו בשלב 3


@app.get("/orders/{order_id}")
def get_order(order_id: str):
    if order_id not in ORDERS:
        raise HTTPException(status_code=404, detail="Order not found")
    return ORDERS[order_id]


server, BASE = start_server(app)
print("✓ השרת רץ ברקע בכתובת", BASE)

<div dir="rtl">

## שלב 1 - חמש בקשות, חמישה קודים

הפונקציה `show(method, path, ...)` (כבר מוגדרת) שולחת בקשה אמיתית לשרת ומדפיסה את התגובה: שורת סטטוס, סוג התוכן וגוף. תהיינה חמש בקשות:

1. `GET /health`
2. `GET /products?category=Tech&limit=2`
3. `GET /products?limit=0`
4. `GET /products/P001` (שימו לב: `find_product` עוד לא נכתבה)
5. `GET /no-such-page`

</div>

<div dir="rtl">

**לפני שמריצים:** איזה קוד סטטוס יחזור לכל אחת מחמש הבקשות? ולכל קוד - מי "אשם": הבקשה (הלקוח) או השרת?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
show("GET", "/health")
print()
show("GET", "/products", params={"category": "Tech", "limit": 2})
print()
show("GET", "/products", params={"limit": 0})
print()
show("GET", "/products/P001")
print()
show("GET", "/no-such-page")

<div dir="rtl">

## שלב 1ב - הספרה הראשונה אומרת הכול

ממשו `status_family(code)`: מקבלת קוד סטטוס (מספר) ומחזירה טקסט: `"success"` (2xx), `"redirect"` (3xx), `"client_error"` (4xx), `"server_error"` (5xx). לקוח (או סוכן!) שמקבל תגובה צריך לדעת בדיוק את זה: על 4xx לא מנסים שוב את אותה בקשה; על 5xx - אולי מנסים מאוחר יותר.

**כלי:** חילוק שלם `code // 100` נותן את הספרה הראשונה (`404 // 100` הוא `4`).

</div>

In [ ]:
def status_family(code):
    """code: an HTTP status code (int). Returns "success" (2xx), "redirect" (3xx),
    "client_error" (4xx) or "server_error" (5xx).
    e.g. status_family(404) -> "client_error"
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 1ב
def check_a1():
    expected = {200: "success", 201: "success", 202: "success", 301: "redirect",
                404: "client_error", 422: "client_error", 429: "client_error", 500: "server_error", 503: "server_error"}
    for code, family in expected.items():
        got = status_family(code)
        assert got == family, f"status_family({code}): ציפינו ל-{family!r}, התקבל {got!r} (הספרה הראשונה: {code // 100})"
    return True

check_a1() and print("✓ שלב 1ב:", {c: status_family(c) for c in (200, 404, 422, 500)})

<div dir="rtl">

**(א)** בשלב 1 קיבלנו גם 422 וגם 500, ושניהם "שגיאה". מה ההבדל ביניהם במונחי **מי אשם** ומה הלקוח צריך לעשות בכל אחד? ולמה 404 עדיף על 200 עם הודעת "לא נמצא" בגוף?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 2 - קוד הצד של השרת: 404

עכשיו אתם הצד של השרת. בלי `find_product` כל בקשה למוצר קורסת (500) - ראיתם בשלב 1. ממשו אותה: מקבלת `product_id` ומחזירה את המוצר; **אם אין מוצר כזה - מחזירה ללקוח 404**.

**כלים:** `PRODUCTS.get(product_id)` מחזיר את המוצר, או `None` אם אין. כדי להחזיר ללקוח קוד שגיאה: `raise HTTPException(status_code=404, detail="Product not found")` (ה-`raise` עוצר את הפונקציה ו-FastAPI הופך אותה לתגובת 404).

</div>

In [ ]:
def find_product(product_id):
    """product_id: text like "P001". PRODUCTS is a dict: product_id -> product (a dict).
    Returns the product dict. If there is no such product: raise HTTPException with status_code=404.
    e.g. find_product("P001")["name"] -> "Desk Lamp Basic"
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 2 (הבדיקה פונה לשרת האמיתי)
def check_a2():
    r = requests.get(BASE + "/products/P001", timeout=10)
    assert r.status_code != 500, "GET /products/P001: התקבל 500 - השרת קרס. כנראה find_product עדיין לא מומשה או שיש בה שגיאה"
    assert r.status_code == 200, f"GET /products/P001: ציפינו ל-200, התקבל {r.status_code}"
    assert r.json()["name"] == "Desk Lamp Basic", "find_product: צריך להחזיר את המוצר P001 (Desk Lamp Basic)"
    r = requests.get(BASE + "/products/NOPE", timeout=10)
    assert r.status_code != 500, "GET /products/NOPE: התקבל 500 - מוצר שלא קיים צריך להחזיר 404, לא להקריס את השרת (בדקו if ... is None)"
    assert r.status_code == 404, f"GET /products/NOPE: ציפינו ל-404 (מוצר לא קיים), התקבל {r.status_code}"
    assert "detail" in r.json(), "find_product: העבירו detail=... ל-HTTPException, כדי שהלקוח יידע מה קרה"
    return True

check_a2() and print("✓ שלב 2: P001 מחזיר 200, ו-NOPE מחזיר 404")

<div dir="rtl">

## שלב 3 - יצירת הזמנה: 201 ו-422

`POST /orders` יוצר הזמנה. גוף הבקשה הוא JSON כמו `{"product_id": "P001", "quantity": 2}`. ממשו `create_order(order)`: מקבלת את הגוף (`order.product_id`, `order.quantity` - אובייקט של `OrderIn`) ו**מחזירה מילון של ההזמנה החדשה** עם ארבעה מפתחות: `order_id`, `product_id`, `quantity`, `total` (= כמות × מחיר המוצר). את ההזמנה שומרים ב-`ORDERS` לפי ה-`order_id`.

כמה דברים כבר קורים בלי שתכתבו אותם: **201** מגיע מ-`status_code=201` בהגדרת הכתובת; **422** על `quantity` שאינו חיובי מגיע מהמודל `OrderIn`, לפני שהפונקציה שלכם בכלל רצה. מה שנשאר לכם: מוצר שלא קיים ← 404.

**כלים:** `find_product(order.product_id)` (אתם כבר כתבתם!) מחזירה את המוצר, או מקימה 404 לבד. מספר סידורי להזמנה: `order_id = "W" + str(len(ORDERS) + 1)` (W1, W2, ...). `round(x, 2)` מעגל לשתי ספרות.

</div>

In [ ]:
def create_order(order):
    """order: an OrderIn (order.product_id, order.quantity). Creates a new order and returns it as a dict
    with the keys order_id, product_id, quantity, total (= quantity * the product's unit_price, rounded to 2).
    The order is also stored in ORDERS (order_id -> order dict). An unknown product must give 404.
    e.g. create_order(OrderIn(product_id="P001", quantity=2))["total"] -> 12.68
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 3
def check_a3():
    r = requests.post(BASE + "/orders", json={"product_id": "P001", "quantity": 2}, timeout=10)
    assert r.status_code != 500, "POST /orders: התקבל 500 - השרת קרס. כנראה create_order עדיין לא מומשה או שיש בה שגיאה"
    assert r.status_code == 201, f"POST /orders: ציפינו ל-201 (נוצר), התקבל {r.status_code}"
    o = r.json()
    for key in ("order_id", "product_id", "quantity", "total"):
        assert key in o, f"create_order: חסר המפתח {key!r} בהזמנה שהוחזרה"
    assert o["quantity"] == 2 and o["product_id"] == "P001", "create_order: product_id ו-quantity צריכים להיות כמו בבקשה"
    assert abs(o["total"] - 12.68) < 0.01, f"create_order: total צריך להיות 2 * 6.34 = 12.68, התקבל {o['total']}"
    o2 = requests.post(BASE + "/orders", json={"product_id": "P001", "quantity": 1}, timeout=10).json()
    assert o2["order_id"] != o["order_id"], "create_order: לכל הזמנה צריך להיות order_id שונה"
    g = requests.get(BASE + "/orders/" + o["order_id"], timeout=10)
    assert g.status_code == 200 and g.json()["order_id"] == o["order_id"], "create_order: צריך לשמור את ההזמנה ב-ORDERS (ואז GET /orders/<id> מחזיר 200)"
    r = requests.post(BASE + "/orders", json={"product_id": "NOPE", "quantity": 1}, timeout=10)
    assert r.status_code == 404, f"POST /orders עם מוצר שלא קיים: ציפינו ל-404, התקבל {r.status_code}"
    r = requests.post(BASE + "/orders", json={"product_id": "P001", "quantity": 0}, timeout=10)
    assert r.status_code == 422, f"POST /orders עם quantity=0: ציפינו ל-422 (החוזה), התקבל {r.status_code}"
    return True

check_a3() and print("✓ שלב 3: 201 להזמנה חדשה, 404 למוצר שלא קיים, 422 לכמות לא חוקית")

<div dir="rtl">

**לפני שמריצים:** שלוש בקשות `POST /orders`: (1) `P001` כמות 3; (2) `P001` כמות 0; (3) `NOPE` כמות 1. איזה קוד יחזור לכל אחת, ומי מייצר אותו - **הקוד שכתבתם**, או **FastAPI לפי החוזה**?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
show("POST", "/orders", json={"product_id": "P001", "quantity": 3})
print()
show("POST", "/orders", json={"product_id": "P001", "quantity": 0})
print()
show("POST", "/orders", json={"product_id": "NOPE", "quantity": 1})

<div dir="rtl">

**(ב)** שני קודי שגיאה, 422 ו-404, הגיעו משני מקומות שונים בשירות. מה מייצר כל אחד? ולמה כדאי שגבולות כמו `quantity > 0` יהיו מוגדרים **פעם אחת במודל** (ולא בבדיקה ידנית בתוך כל פונקציה)? (שקף "מודל = חוזה")

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 4 - החוזה: `openapi.json`

FastAPI כתב **לבד** את קובץ החוזה, ואפשר לקרוא אותו מהכתובת `/openapi.json`. (ב-Colab אין דפדפן שפונה לשרת הרץ ברקע, ולכן `/docs` - Swagger UI - לא נפתח כאן. הוא מציג בדיוק את אותו קובץ, ובמעבדה בכיתה תראו אותו במחשב שלכם.) בקובץ, תחת `paths`: כל כתובת ← כל שיטה ← `responses`: אילו קודי סטטוס **מתועדים**.

</div>

<div dir="rtl">

**לפני שמריצים:** אילו קודי סטטוס לדעתכם מתועדים בחוזה עבור `GET /products/{product_id}`? ועבור `POST /orders`? ועבור `GET /orders/{order_id}`?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
spec = requests.get(BASE + "/openapi.json", timeout=10).json()
print("גרסת OpenAPI:", spec["openapi"], "| כותרת:", spec["info"]["title"])
print()
for path, methods in spec["paths"].items():
    for method, info in methods.items():
        print(f"{method.upper():5} {path:26} תגובות מתועדות: {list(info['responses'])}")

limit_param = [p for p in spec["paths"]["/products"]["get"]["parameters"] if p["name"] == "limit"][0]
print()
print("הפרמטר limit בחוזה:", limit_param["schema"])

<div dir="rtl">

## שלב 5 - קוראים את החוזה בקוד: `list_endpoints`

המבנה: `spec["paths"]` הוא מילון: מפתח = כתובת (`"/products"`), וערך = מילון של שיטות (`"get"`, `"post"`) לתיאור. ממשו `list_endpoints(spec)`: מחזירה רשימה **ממוינת** של כל נקודות הקצה כ-tuple של `(METHOD, path)` - השיטה באותיות גדולות. למשל `("GET", "/products")`.

**איך:** לולאה בתוך לולאה: `for path, methods in spec["paths"].items():` ואז `for method in methods:`. `method.upper()` הופך לאותיות גדולות. `sorted(רשימה)` ממיין רשימת tuple.

</div>

In [ ]:
def list_endpoints(spec):
    """spec: the OpenAPI document (a dict). Returns a SORTED list of (METHOD, path) tuples, METHOD in capitals.
    e.g. for a spec with paths {"/a": {"get": ..., "post": ...}} -> [("GET", "/a"), ("POST", "/a")]
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 5
def check_a5():
    small = {"paths": {"/b": {"delete": {}}, "/a": {"post": {}, "get": {}}}}
    got = [tuple(x) for x in list_endpoints(small)]
    assert got == [("DELETE", "/b"), ("GET", "/a"), ("POST", "/a")], \
        f"list_endpoints: על החוזה הקטן ציפינו ל-[('DELETE', '/b'), ('GET', '/a'), ('POST', '/a')] (ממוין, שיטה באותיות גדולות), התקבל {got}"
    live = [tuple(x) for x in list_endpoints(spec)]
    assert len(live) == 5 and ("POST", "/orders") in live and ("GET", "/products/{product_id}") in live, \
        "list_endpoints: על החוזה של השירות צריכות להיות 5 נקודות קצה, למשל ('POST', '/orders')"
    return True

check_a5() and print("✓ שלב 5:", list_endpoints(spec))

<div dir="rtl">

## שלב 6 - בדיקה שנגזרת מהחוזה: `check_contract`

זה הרעיון המרכזי של החלק: **החוזה קובע מה אמור לקרות, והבדיקה מוודאת שהשרת עומד בו.** נבנה כלי שעובר על **כל** נקודות הקצה בחוזה, קורא לכל אחת באמת, ובודק: **האם קוד הסטטוס שהשרת החזיר בפועל מופיע ברשימת הקודים המתועדים?** (כלים כמו Schemathesis עושים זאת בגדול; זה הרעיון של "בדיקות אוטומטיות מהחוזה" מההרצאה.)

כלי העזר שכבר מוגדרים (קראו, אל תשנו): `urls_for(path)` ממלאת את `{product_id}` בערכים לדוגמה (גם ערך שקיים וגם `NOPE`), ו-`SAMPLE_BODIES` נותן גוף לדוגמה ל-`POST /orders`.

</div>

In [ ]:
SAMPLES = {"product_id": ["P001", "NOPE"], "supplier_id": ["S01", "NOPE"], "order_id": ["O1"]}
SAMPLE_BODIES = {("POST", "/orders"): {"product_id": "P001", "quantity": 2}}

def urls_for(path):
    """'/products/{product_id}' -> ['/products/P001', '/products/NOPE'];  a path without {...} -> [path]"""
    m = re.search(r"\{(\w+)\}", path)
    if m is None:
        return [path]
    return [path.replace(m.group(0), value) for value in SAMPLES[m.group(1)]]

print(urls_for("/products/{product_id}"), urls_for("/health"))

<div dir="rtl">

ממשו `check_contract(spec, base)`: לכל `(method, path)` מ-`list_endpoints(spec)` ולכל כתובת מ-`urls_for(path)`: שלחו בקשה (`requests.request(method, base + url, json=SAMPLE_BODIES.get((method, path)), timeout=10)`) ובדקו אם `r.status_code` נמצא בקודים המתועדים. מחזירה **רשימת בעיות** (טקסטים), אחת לכל קריאה שהקוד שלה **לא** בחוזה. רשימה ריקה = השרת עומד בחוזה.

**שימו לב:** הקודים בחוזה הם **טקסט** (`"200"`), וקוד הסטטוס בתגובה הוא **מספר** (`200`) - צריך `str(r.status_code)`. הקודים המתועדים: `list(spec["paths"][path][method.lower()]["responses"])`.

</div>

In [ ]:
def check_contract(spec, base):
    """For every (method, path) in the spec and every url in urls_for(path): send the request
    (json=SAMPLE_BODIES.get((method, path))) and check that the status code the server REALLY returned
    is one of the codes the spec documents for that endpoint (spec["paths"][path][method.lower()]["responses"]).
    Returns a list of problem texts, one per call whose status code is NOT in the spec. [] = server matches its contract.
    e.g. "GET /orders/O1 -> 404, spec lists ['200', '422']"
    """
    problems = []
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - שלב 6
def check_a6():
    lying = {"paths": {"/health": {"get": {"responses": {"404": {}}}}}}      # חוזה שמתעד רק 404, אבל /health מחזיר 200
    honest = {"paths": {"/health": {"get": {"responses": {"200": {}}}}}}
    assert len(check_contract(honest, BASE)) == 0, "check_contract: ל-/health מתועד 200 והשרת מחזיר 200 - לא אמורה להיות בעיה (השוו str(r.status_code) למפתחות הטקסט)"
    assert len(check_contract(lying, BASE)) == 1, "check_contract: החוזה מתעד רק 404 אבל השרת מחזיר 200 - צריך להחזיר בעיה אחת"
    post_only = {"paths": {"/orders": {"post": {"responses": {"201": {}}}}}}
    assert len(check_contract(post_only, BASE)) == 0, "check_contract: ל-POST /orders צריך לשלוח את הגוף לדוגמה (json=SAMPLE_BODIES.get((method, path))) - בלי גוף השרת מחזיר 422 במקום 201"
    problems = check_contract(spec, BASE)
    assert len(problems) <= 1, f"check_contract: נמצאו {len(problems)} בעיות, וציפינו לאחת לכל היותר. בדקו: מפתחות responses הם טקסט והקוד מספר: str(r.status_code). הבעיות: {problems}"
    assert len(problems) == 1 and "orders" in problems[0], "check_contract: ציפינו שהכלי ימצא אי-התאמה אחת, ב-GET /orders/... (הריצו את התא שאחרי הבדיקה כדי לראות)"
    return True

check_a6() and print("✓ שלב 6: הכלי מזהה שרת שלא עומד בחוזה שלו")

<div dir="rtl">

**לפני שמריצים על השירות האמיתי:** האם השרת עומד בחוזה שלו? כמה אי-התאמות תמצא `check_contract`, ובאיזו נקודת קצה?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
problems = check_contract(spec, BASE)
if problems:
    print(f"נמצאו {len(problems)} אי-התאמות בין החוזה לשרת:")
    for p in problems:
        print("  ⚠", p)
else:
    print("✓ השרת עומד בחוזה שלו")

<div dir="rtl">

**(ג)** מה אי-ההתאמה שהכלי מצא? אילו שתי דרכים יש לתקן אותה, ומי צריך להחליט ביניהן? ומה זה אומר על המשפט "FastAPI מייצר את החוזה מהקוד" - האם אפשר להסתמך על `openapi.json` בלי לבדוק? (רמז: שקף "מה שבחוזה: גם תגובות שגויות")

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ב' - SQL וסכמה: שואלים את הנתונים ושומרים על קלט בטוח (12 דק')

## הרקע

מאחורי ה-API יושב **מסד נתונים**. כאן: SQLite (אותו SQL של מסדי נתונים אחרים), עם נתוני החנות מהמעבדות הקודמות: 200 מוצרים, 3,000 הזמנות, 8 ספקים. **סכמה** = אילו טבלאות, אילו עמודות, ואיך הן מתחברות: **מפתח ראשי 🔑** מזהה שורה, **מפתח זר 🔗** מצביע על טבלה אחרת. את שאילתת ה-SQL קוראים כמשפט:

| חלק | משמעות |
|---|---|
| `SELECT` | מה להציג |
| `FROM` | מאיזו טבלה |
| `JOIN ... ON/USING` | לחבר טבלה נוספת לפי מפתח משותף |
| `WHERE` | רק אם... |
| `GROUP BY` | לקבץ לפי |
| `ORDER BY` | למיין (`DESC` = מהגבוה לנמוך) |
| `LIMIT` | כמה שורות |

הריצו את התא: הוא יוצר את הסכמה (קראו את ה-`CREATE TABLE`), טוען את הנתונים, ומגדיר `run_sql(sql, params)` שמריצה שאילתה ומחזירה טבלת pandas.

</div>

In [ ]:
SCHEMA = """
CREATE TABLE suppliers (
    supplier_id    TEXT PRIMARY KEY,                               -- 🔑 מפתח ראשי
    name           TEXT NOT NULL,
    lead_time_days INTEGER NOT NULL,
    on_time_rate   REAL NOT NULL
);
CREATE TABLE products (
    product_id  TEXT PRIMARY KEY,                                  -- 🔑
    name        TEXT NOT NULL,
    category    TEXT NOT NULL,
    unit_cost   REAL NOT NULL,
    unit_price  REAL NOT NULL,
    supplier_id TEXT NOT NULL REFERENCES suppliers(supplier_id)    -- 🔗 מפתח זר: מצביע על suppliers
);
CREATE TABLE orders (
    order_id     TEXT PRIMARY KEY,                                 -- 🔑
    order_date   TEXT NOT NULL,
    customer_id  TEXT NOT NULL,
    product_id   TEXT NOT NULL REFERENCES products(product_id),    -- 🔗 מפתח זר: מצביע על products
    product_name TEXT NOT NULL,
    quantity     INTEGER NOT NULL,
    unit_price   REAL NOT NULL
);
"""
con = sqlite3.connect(":memory:", check_same_thread=False)         # מסד בזיכרון (נעלם כשמפעילים מחדש)
con.execute("PRAGMA foreign_keys = ON")                            # ב-SQLite צריך להפעיל במפורש, אחרת REFERENCES אינו נאכף
con.executescript(SCHEMA)
tables = {name: pd.read_csv(f"{name}.csv") for name in ("suppliers", "products", "orders")}
for name, df in tables.items():
    df.to_sql(name, con, if_exists="append", index=False)

def run_sql(sql, params=()):
    """מריצה שאילתה ומחזירה טבלת pandas. params: הערכים שימולאו במקום סימני ה-? בשאילתה."""
    return pd.read_sql_query(sql, con, params=params)

def run_sql_checked(sql, params=(), what="השאילתה"):
    """כמו run_sql, אבל אם SQLite מחזיר שגיאה - מציגה אותה בהודעה ברורה (לתאי הבדיקה)."""
    try:
        return run_sql(sql, params)
    except Exception as e:
        raise AssertionError(f"{what}: SQLite החזיר שגיאה: {e}")

print({name: len(df) for name, df in tables.items()}, "שורות בכל טבלה")
run_sql("SELECT * FROM products LIMIT 3")

<div dir="rtl">

## שלב 1 - שאילתה פשוטה: הספקים הכי פחות דייקנים

ממשו `q_slow_suppliers()`: מחזירה **טקסט** של שאילתת SQL (מחרוזת). השאילתה מחזירה את העמודות `name` ו-`on_time_rate` של **שלושת הספקים עם שיעור האספקה בזמן (`on_time_rate`) הנמוך ביותר**, מהנמוך לגבוה.

**כלים:** `SELECT` + `FROM` + `ORDER BY` (`ASC` = מהנמוך לגבוה, ברירת המחדל) + `LIMIT`. שימו לב: מחזירים את *הטקסט*, לא מריצים אותו - את ההרצה עושה `run_sql`.

</div>

In [ ]:
def q_slow_suppliers():
    """Returns SQL text (a string). The query returns the columns name and on_time_rate
    of the 3 suppliers with the LOWEST on_time_rate, the lowest first.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ב' שלב 1
def check_b1():
    df = run_sql_checked(q_slow_suppliers(), what="q_slow_suppliers")
    assert list(df.columns) == ["name", "on_time_rate"], f"q_slow_suppliers: צריך להחזיר בדיוק את העמודות name ו-on_time_rate, התקבל {list(df.columns)}"
    assert list(df["name"]) == ["Supplier A", "Supplier H", "Supplier D"], f"q_slow_suppliers: ציפינו ל-Supplier A, H, D (מהנמוך לגבוה), התקבל {list(df['name'])}"
    return True

check_b1() and print("✓ ב' שלב 1:")
print(run_sql(q_slow_suppliers()))

<div dir="rtl">

## שלב 2 - JOIN ו-GROUP BY: הכנסה לכל קטגוריה

**הכנסה = כמות × מחיר** (כמו תמיד). אבל **קטגוריה** נמצאת בטבלת `products`, והכמות והמחיר - בטבלת `orders`. צריך **JOIN**: להצמיד לכל הזמנה את פרטי המוצר שלה לפי `product_id`.

ממשו `q_category_revenue()`: מחזירה טקסט של שאילתה עם העמודות `category` ו-`revenue` (סכום `quantity * unit_price` של ההזמנות), **לכל קטגוריה**, מהגבוה לנמוך.

**כלים:** `FROM orders o JOIN products p USING (product_id)` (האותיות `o` ו-`p` הן כינויים קצרים). **שימו לב:** `unit_price` קיים בשתי הטבלאות! כתבו `o.unit_price` (המחיר שבהזמנה) - בלי הקידומת SQLite יחזיר שגיאה `ambiguous column name`. אחר כך `GROUP BY p.category`, `SUM(...) AS revenue`, ו-`ORDER BY revenue DESC`.

</div>

In [ ]:
def q_category_revenue():
    """Returns SQL text. Columns: category, revenue (= SUM(quantity * unit_price) of the ORDERS) - one row per category,
    highest revenue first. Needs a JOIN between orders and products.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ב' שלב 2 (משווים לחישוב בלתי-תלוי ב-pandas: "סכום מוכר")
def check_b2():
    o, p = tables["orders"], tables["products"]
    m = o.merge(p[["product_id", "category"]], on="product_id")
    truth = (m["quantity"] * m["unit_price"]).groupby(m["category"]).sum().sort_values(ascending=False)
    df = run_sql_checked(q_category_revenue(), what="q_category_revenue (רמז: unit_price קיים בשתי הטבלאות - כתבו o.unit_price)")
    assert list(df.columns) == ["category", "revenue"], f"q_category_revenue: צריך את העמודות category ו-revenue, התקבל {list(df.columns)}"
    assert len(df) == 5, f"q_category_revenue: יש 5 קטגוריות, התקבלו {len(df)} שורות (GROUP BY p.category)"
    assert list(df["category"]) == list(truth.index), f"q_category_revenue: הסדר צריך להיות {list(truth.index)} (מהגבוה לנמוך), התקבל {list(df['category'])}"
    assert all(abs(df["revenue"].values - truth.values) < 0.5), "q_category_revenue: הסכומים לא תואמים. בדקו שהסכום הוא SUM(o.quantity * o.unit_price) ושה-JOIN לפי product_id"
    return True

check_b2() and print("✓ ב' שלב 2:")
print(run_sql(q_category_revenue()).round(0))

<div dir="rtl">

## שלב 3 - שאילתה עם פרמטר (`?`): חמשת המוצרים המובילים

אותה משימת עוגן ממפגש 5 - **המוצרים עם ההכנסה הגבוהה ביותר** - אבל עכשיו במסד, וב-**N כפרמטר**: השירות צריך לדעת להחזיר 3, או 5, או 10 מוצרים. את המספר לא מדביקים לטקסט השאילתה, אלא כותבים `LIMIT ?`, והערך מועבר **בנפרד** כשמריצים: `run_sql(sql, (5,))`.

ממשו `q_top_products()`: טקסט שאילתה עם העמודות `product_name` ו-`revenue` (מעוגל ל-2 ספרות: `ROUND(SUM(...), 2)`), לכל מוצר לפי `product_name` (בלי JOIN - השם נמצא גם ב-`orders`), מהגבוה לנמוך, עם `LIMIT ?`.

מה נחשב נכון: **Plant Pot Plus עם 33,595.62** ראשון (כמו במפגש 5).

</div>

In [ ]:
def q_top_products():
    """Returns SQL text with ONE placeholder (?) for the number of rows. Columns: product_name,
    revenue (= ROUND(SUM(quantity * unit_price), 2)), one row per product_name, highest revenue first.
    It will be used like: run_sql(q_top_products(), (5,))
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ב' שלב 3
def check_b3():
    sql = q_top_products()
    assert "?" in sql, "q_top_products: חסר סימן ? - את N מעבירים כפרמטר (LIMIT ?), לא מדביקים לטקסט"
    o = tables["orders"]
    truth = (o["quantity"] * o["unit_price"]).groupby(o["product_name"]).sum().sort_values(ascending=False)
    df5 = run_sql_checked(sql, (5,), what="q_top_products")
    assert list(df5.columns) == ["product_name", "revenue"], f"q_top_products: צריך את העמודות product_name ו-revenue, התקבל {list(df5.columns)}"
    assert len(df5) == 5, "q_top_products: עם הפרמטר 5 צריכות לחזור 5 שורות (LIMIT ?)"
    assert list(df5["product_name"]) == list(truth.index[:5]), "q_top_products: הסדר/המוצרים לא נכונים. ודאו GROUP BY product_name ו-ORDER BY revenue DESC"
    assert df5["product_name"][0] == "Plant Pot Plus" and abs(df5["revenue"][0] - 33595.62) < 0.01, "q_top_products: המוצר הראשון צריך להיות Plant Pot Plus עם 33,595.62"
    assert len(run_sql_checked(sql, (2,), what="q_top_products")) == 2, "q_top_products: עם הפרמטר 2 צריכות לחזור 2 שורות - N חייב להיות הפרמטר ולא מספר קבוע"
    return True

check_b3() and print("✓ ב' שלב 3:")
print(run_sql(q_top_products(), (5,)))

<div dir="rtl">

## שלב 4 - JOIN שמכפיל שורות

אחת הטעויות הנפוצות ביותר ב-SQL שכלי AI כותב: **JOIN שבו שורה אחת מתאימה לכמה שורות - והנתונים מוכפלים בשקט.** השאילתה רצה, התוצאה נראית סבירה, והמספר שגוי. הבדיקה: **סופרים שורות לפני ואחרי** ה-JOIN (ומשווים לסכום שמכירים).

</div>

<div dir="rtl">

**לפני שמריצים:** כמה שורות יחזיר `orders JOIN products` (לפי product_id)? וכמה שורות יחזיר `suppliers JOIN products` (לפי supplier_id)? באיזה מהשניים השורות מוכפלות, ולמה? (רמז: מי "אחד" ומי "רבים" בכל חיבור?)

**התחזית שלי:** _(כתבו כאן)_

</div>

<div dir="rtl">

ממשו `join_multiplies(base_sql, join_sql)`: מקבלת **שתי שאילתות** (טקסט): השאילתה הבסיסית והשאילתה אחרי ה-JOIN. מחזירה `True` אם ה-JOIN החזיר **יותר שורות** מהבסיס (= הכפלה), אחרת `False`.

**כלים:** `run_sql(sql)` מחזירה טבלה, ו-`len(טבלה)` נותן את מספר השורות.

**שימו לב לגבול של הבדיקה:** היא מזהה רק **גידול** בשורות. JOIN יכול גם **לאבד** שורות: `INNER JOIN` משמיט שורות בלי התאמה בטבלה השנייה (ובדיוק לזה משתמשים ב-`LEFT JOIN`). בפרויקט אמיתי בודקים שמספר השורות לא השתנה בשני הכיוונים.

</div>

In [ ]:
def join_multiplies(base_sql, join_sql):
    """base_sql, join_sql: two SQL texts. Runs both. Returns True if the JOIN query returned MORE rows than the base query.
    e.g. join_multiplies("SELECT * FROM suppliers", "SELECT * FROM suppliers s JOIN products p USING (supplier_id)") -> True
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ב' שלב 4
def check_b4():
    assert join_multiplies("SELECT * FROM suppliers", "SELECT * FROM suppliers s JOIN products p USING (supplier_id)") is True, \
        "join_multiplies: 8 ספקים הפכו ל-200 שורות אחרי JOIN - צריך להחזיר True"
    assert join_multiplies("SELECT * FROM orders", "SELECT * FROM orders o JOIN products p USING (product_id)") is False, \
        "join_multiplies: 3,000 הזמנות נשארו 3,000 אחרי JOIN - אין הכפלה, צריך להחזיר False"
    return True

check_b4() and print("✓ ב' שלב 4: הכפלה מזוהה")

In [ ]:
# מה ההכפלה עושה לסכום? זמן אספקה כולל של הספקים:
print("סכום lead_time_days, ספקים בלבד:     ", run_sql("SELECT SUM(lead_time_days) AS total FROM suppliers")["total"][0])
print("סכום lead_time_days, אחרי JOIN מוצרים:", run_sql("SELECT SUM(s.lead_time_days) AS total FROM suppliers s JOIN products p USING (supplier_id)")["total"][0])

<div dir="rtl">

**(א)** מה קרה לסכום אחרי ה-JOIN, ולמה? איך הייתם מגלים את זה אילו לא הכרתם את התשובה הנכונה? (רמז: שקף "איך מאמתים שאילתה": ספירת שורות, סכום מוכר, דוגמה קטנה)

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 5 - הערך הולך ב-`?`, לא לתוך הטקסט (SQL injection)

שאילתה שבונים בהדבקת טקסט (`f"... WHERE name = '{name}'"`) היא חור אבטחה: מי שמקליד שם מוצר **זדוני** משנה את משמעות השאילתה. כך כלי AI עלול לכתוב קוד "שעובד" אבל פרוץ. התיקון שכבר ראיתם בשקף: `?` - הערך מועבר **בנפרד**, והמסד מתייחס אליו כנתון ולא כקוד.

</div>

In [ ]:
def find_products_unsafe(name):
    sql = f"SELECT product_id, name FROM products WHERE name = '{name}'"      # הערך מודבק לתוך הטקסט!
    return run_sql(sql)

print(len(find_products_unsafe("Desk Lamp Basic")), "שורה לשם רגיל")

<div dir="rtl">

**לפני שמריצים:** מה יקרה אם נקליד כשם המוצר את הטקסט `x' OR '1'='1` - כמה שורות יחזרו מ-`find_products_unsafe`? (חשבו מה הטקסט הופך להיות אחרי ההדבקה.)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
print(len(find_products_unsafe("x' OR '1'='1")), "שורות לשם הזדוני")

<div dir="rtl">

ממשו `find_products_safe(name)`: מחזירה רשימה של השורות (כל שורה: `(product_id, name)`) של המוצרים ששמם שווה ל-`name` - **עם `?`**. `con.execute(sql, (value,)).fetchall()` מריצה שאילתה עם פרמטר ומחזירה רשימת שורות. שימו לב לפסיק ב-`(value,)`: tuple של איבר אחד.

</div>

In [ ]:
def find_products_safe(name):
    """Returns a list of rows (product_id, name) of the products whose name equals `name`.
    The value must go in as a PARAMETER (?), never pasted into the SQL text.
    e.g. find_products_safe("Desk Lamp Basic") -> [("P001", "Desk Lamp Basic")]
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ב' שלב 5
def check_b5():
    assert [tuple(r) for r in find_products_safe("Desk Lamp Basic")] == [("P001", "Desk Lamp Basic")], "find_products_safe: לשם 'Desk Lamp Basic' צריכה לחזור שורה אחת: ('P001', 'Desk Lamp Basic')"
    assert len(find_products_safe("x' OR '1'='1")) == 0, "find_products_safe: השם הזדוני צריך להיחשב סתם שם שלא קיים - 0 שורות. ודאו שהערך עובר כפרמטר (?) ולא בהדבקה"
    try:
        assert len(find_products_safe("O'Brien")) == 0, "find_products_safe: שם שלא קיים צריך להחזיר 0 שורות"
    except sqlite3.Error:
        raise AssertionError("find_products_safe: שם עם גרש (O'Brien) שבר את השאילתה - זה סימן שהערך מודבק לטקסט. השתמשו ב-?")
    try:
        src = inspect.getsource(find_products_safe)
        assert "?" in src, "find_products_safe: אין ? בקוד - הערך צריך לעבור כפרמטר"
    except (OSError, TypeError):
        pass
    return True

check_b5() and print("✓ ב' שלב 5: השם הזדוני לא מחזיר דבר")

<div dir="rtl">

**(ב)** למה `?` מונע את ההתקפה? מה המסד עושה אחרת עם הערך כשהוא מועבר בנפרד? ובאיזה שלב בשירות מ-חלק א' כדאי לשים לב לזה כשכלי AI כותב לכם נקודת קצה שניגשת למסד?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ג' - משלבים: API ↔ מסד נתונים ↔ AI (10 דק')

## הרקע

בשירות אמיתי הממשק לא נוגע במסד: **ה-API הוא השער היחיד** (ולידציה, חוזה, הרשאות). עד עכשיו ה-API שלנו החזיר נתונים מזיכרון (`PRODUCTS`), ובחלק ב' כתבתם שאילתות. הגיע הזמן לחבר: **API שהנתונים שלו מגיעים מהמסד**. שלושה צעדים:

1. **חיבור** - פונקציה אחת שמחברת את ה-API לשאילתה שכתבתם.
2. **בדיקה מקצה לקצה** (end-to-end): לקוח ← HTTP ← API ← SQL ← חזרה. ואז - **בודקים את הבדיקה**: האם היא תופסת גרסאות שגויות?
3. **צעד אמיתי עם AI**: מבקשים מ-Gemini לכתוב נקודת קצה לפי חוזה נתון, ובודקים אותה **בכלי שבניתם בחלק א'**.

הקוד שמגדיר שרת שני (על אותו מסד של חלק ב'). את שתי הפונקציות שהוא קורא להן - `top_products_from_db` ו-`get_supplier` - עוד לא כתבתם:

</div>

In [ ]:
DB_LOCK = threading.Lock()      # חיבור המסד משותף לתהליכוני השרת: בקשה אחת בכל רגע

app2 = FastAPI(title="Store API v2 (with DB)", version="2.0.0")


@app2.get("/reports/top-products")
def top_products(n: int = Query(5, ge=1, le=50)):
    with DB_LOCK:
        return top_products_from_db(n)            # שלב 1: אתם כותבים


@app2.get("/suppliers/{supplier_id}", responses={404: {"description": "Supplier not found"}})
def supplier(supplier_id: str):
    with DB_LOCK:
        return get_supplier(supplier_id)          # שלב 3: Gemini כותב


server2, BASE2 = start_server(app2)
print("✓ שרת מס' 2 רץ ברקע בכתובת", BASE2)

<div dir="rtl">

## שלב 1 - חיבור: `top_products_from_db`

ממשו `top_products_from_db(n)`: מחזירה **רשימה של מילונים** (`{"product_name": ..., "revenue": ...}`) של n המוצרים המובילים - **מהמסד**, בעזרת מה שכבר כתבתם בחלק ב'. זו הפונקציה היחידה שמחברת בין ה-API ל-SQL.

**כלים:** `run_sql(q_top_products(), (n,))` מחזירה טבלה. `טבלה.to_dict("records")` הופכת טבלה לרשימת מילונים - זה בדיוק מה ש-FastAPI יודע להחזיר כ-JSON.

</div>

In [ ]:
def top_products_from_db(n):
    """Returns a list of dicts [{"product_name": ..., "revenue": ...}, ...]: the n best products, highest revenue first,
    taken from the database with the query you wrote in part B.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ג' שלב 1 (פונה לשרת מס' 2 באמת)
def check_c1():
    r = requests.get(BASE2 + "/reports/top-products", params={"n": 3}, timeout=10)
    assert r.status_code != 500, "GET /reports/top-products: התקבל 500 - השרת קרס. כנראה top_products_from_db לא מומשה או ש-q_top_products מחלק ב' לא עובדת"
    assert r.status_code == 200, f"GET /reports/top-products: ציפינו ל-200, התקבל {r.status_code}"
    rows = r.json()
    assert len(rows) == 3, f"top_products_from_db(n): צריכות לחזור n שורות, חזרו {len(rows)}"
    assert rows[0]["product_name"] == "Plant Pot Plus" and abs(rows[0]["revenue"] - 33595.62) < 0.01, \
        "top_products_from_db: המוצר הראשון צריך להיות Plant Pot Plus עם 33,595.62"
    assert requests.get(BASE2 + "/reports/top-products", params={"n": 0}, timeout=10).status_code == 422, "n=0 צריך להחזיר 422 (זה כבר מוגדר בשרת)"
    return True

check_c1() and print("✓ ג' שלב 1: ה-API מחזיר נתונים מהמסד")
show("GET", "/reports/top-products", base=BASE2, params={"n": 2})

<div dir="rtl">

## שלב 2 - בדיקה מקצה לקצה, ובדיקה של הבדיקה

**מה נחשב נכון** (קריטריוני הקבלה, כמו בחלק ב'): `n=5` ← חמישה מוצרים, מהגבוה לנמוך, **Plant Pot Plus** ראשון עם **33,595.62**.

ממשו `e2e_test()`: קוראת ל-`GET /reports/top-products?n=5` בשרת מס' 2 (`BASE2`) ובודקת עם `assert`: (1) קוד 200; (2) בדיוק 5 שורות; (3) ההכנסות ממוינות מהגבוה לנמוך; (4) המוצר הראשון הוא `Plant Pot Plus` וההכנסה שלו קרובה ל-33595.62 (הפרש קטן מ-0.01). אם הכול תקין - מחזירה `True`.

**כלים:** `requests.get(BASE2 + "/reports/top-products", params={"n": 5}, timeout=10)`, `r.status_code`, `r.json()`, ו-`sorted(רשימה, reverse=True)`.

**ובדיקה של הבדיקה:** בדיקה שמאשרת הכול היא חסרת ערך (שקף "AI כותב בדיקות": *בדיקה שלא ראיתם נכשלת פעם אחת - לא בטוח שהיא בודקת משהו*). לכן אחרי שתכתבו אותה, תא הבדיקה ינסה אותה על **ארבע גרסאות שגויות** של `top_products_from_db` - כמו שכלי AI עלול לכתוב (דומות ל-`ai_chat` ול-`ai_ide` ממפגש 5). **הבדיקה שלכם חייבת להיכשל על כולן.** הכירו אותן:

</div>

In [ ]:
orders_df = tables["orders"]
_full = (orders_df["quantity"] * orders_df["unit_price"]).groupby(orders_df["product_name"]).sum().round(2)

def _rows(series):
    return [{"product_name": name, "revenue": float(value)} for name, value in series.items()]

def buggy_ascending(n):                  # ממיין הפוך: מחזיר את הכי חלשים
    return _rows(_full.sort_values().head(n))

def buggy_alphabetical(n):               # שכח למיין לפי הכנסה: מיון אלפביתי (כמו ai_ide)
    return _rows(_full.sort_index().head(n))

def buggy_by_quantity(n):                # סופר כמות במקום הכנסה (כמו ai_chat)
    return _rows(orders_df.groupby("product_name")["quantity"].sum().sort_values(ascending=False).head(n))

def buggy_one_extra(n):                  # נכון, אבל מחזיר שורה אחת יותר מדי
    return _rows(_full.sort_values(ascending=False).head(n + 1))

BUGGY = {"ascending": buggy_ascending, "alphabetical": buggy_alphabetical,
         "by_quantity": buggy_by_quantity, "one_extra": buggy_one_extra}
print("✓ ארבע גרסאות שגויות הוגדרו:", list(BUGGY))

<div dir="rtl">

**לפני שכותבים:** אילו מארבע הבדיקות (1) קוד 200, (2) בדיוק 5 שורות, (3) מיון, (4) מוצר ראשון והכנסה - תופסת איזו גרסה שגויה? (חשבו על כל גרסה: מה היא מחזירה?)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
def e2e_test():
    """End-to-end test of GET /reports/top-products?n=5 on the server at BASE2.
    Use `assert` for: status 200; exactly 5 rows; revenues sorted high to low;
    first product is "Plant Pot Plus" with revenue within 0.01 of 33595.62. Returns True if all hold.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ג' שלב 2: הבדיקה שלכם עוברת על הגרסה הנכונה, ונכשלת על כל אחת מהשגויות
def check_c2():
    global top_products_from_db
    real = top_products_from_db
    try:
        assert e2e_test() is True, "e2e_test: צריכה להחזיר True כשהכול תקין"
    except AssertionError as e:
        raise AssertionError(f"e2e_test נכשלה על הגרסה הנכונה ({e!r}). ודאו ששלב 1 עבר, ושאין assert שגוי")
    except NotImplementedError:
        raise
    except Exception as e:
        raise AssertionError(f"e2e_test נכשלה על הגרסה הנכונה עם שגיאה ({type(e).__name__}: {e}). ודאו ששלב 1 עבר")
    try:
        for name, bug in BUGGY.items():
            top_products_from_db = bug
            caught = False
            try:
                e2e_test()
            except Exception:
                caught = True
            assert caught, f"e2e_test לא תפסה את הגרסה השגויה {name!r} - חסרה בדיקה שהייתה נכשלת עליה (ראו את ההערה ליד הגרסה למעלה)"
    finally:
        top_products_from_db = real
    return True

check_c2() and print("✓ ג' שלב 2: הבדיקה עוברת על הנכון ותופסת את כל 4 הגרסאות השגויות")

<div dir="rtl">

**(א)** למה `e2e_test` פונה ל-**API** ולא ישירות למסד (כמו `run_sql`)? ומה בדיוק הבדיקה הזו מוכיחה שבדיקת SQL לבדה לא מוכיחה? (שקפים "שכבות ותפקידים")

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

## שלב 3 - צעד אמיתי עם AI: נקודת קצה לפי חוזה

עד עכשיו כתבתם הכול בעצמכם. עכשיו תבקשו מכלי AI לכתוב נקודת קצה **לפי חוזה** - ואתם תבדקו אותה בכלי שבניתם (`check_contract` מחלק א'). זה spec-first בקטן: קודם חוזה, אחר כך קוד, ואז בדיקה.

**החוזה** (מה שה"לקוח" ביקש; זה כבר מוגדר בשרת מס' 2 - הנתיב `GET /suppliers/{supplier_id}`):

```yaml
/suppliers/{supplier_id}:
  get:
    parameters:  supplier_id (path, string)
    responses:
      "200": {supplier_id: string, name: string, lead_time_days: integer, on_time_rate: number}
      "404": {detail: "Supplier not found"}
```

1. פתחו את **עוזר ה-AI של Colab** (כפתור ✨ **Gemini** בסרגל, או "Generate" בתא קוד) - או כל כלי צ'אט שאתם משתמשים בו.
2. בקשו ממנו (העתיקו את הפרומפט כמו שהוא):

```
I have an SQLite connection in a Python variable named `con` (plain sqlite3, default tuple rows)
and a table: suppliers(supplier_id TEXT PRIMARY KEY, name TEXT, lead_time_days INTEGER, on_time_rate REAL).
Write ONE Python function get_supplier(supplier_id) for a FastAPI endpoint:
- query the suppliers table for that supplier_id, using a ? placeholder (no f-strings),
- return a dict with the keys supplier_id, name, lead_time_days, on_time_rate,
- if there is no such supplier, raise fastapi.HTTPException(status_code=404, detail="Supplier not found").
Return only the function and the imports it needs.
```

3. **הדביקו** את הפונקציה שהכלי כתב בתא הבא (בשם `get_supplier`), והריצו.
4. הריצו את תא הבדיקה שאחריו: הוא מפעיל את `check_contract` שלכם על החוזה של שרת מס' 2, ועוד בדיקות תוכן (ספק `S01`, ספק `NOPE`, ושם זדוני).
5. (אם הקוד של הכלי סגר את החיבור `con` או שינה אותו והמסד "נשבר" - הריצו מחדש את תא ההכנה של חלק ב'.)

> **שימו לב:** האם הקוד של הכלי בנוי כמו שביקשתם? **אל תסתפקו ב"זה רץ"** - הכלי שלכם והקריאות בתא הבא בודקים אותו.

</div>

In [ ]:
# הדביקו כאן את הפונקציה שהכלי כתב (בשם get_supplier)
def get_supplier(supplier_id):
    raise NotImplementedError("הדביקו כאן את הקוד שהכלי כתב")

In [ ]:
def supplier_content_problems():
    """בדיקות תוכן לנקודת הקצה של הספק (נתון). מחזיר רשימת בעיות."""
    out = []
    r = requests.get(BASE2 + "/suppliers/S01", timeout=10)
    if r.status_code != 200:
        out.append(f"GET /suppliers/S01 -> {r.status_code} (ציפינו ל-200)")
    else:
        truth = run_sql("SELECT supplier_id, name, lead_time_days, on_time_rate FROM suppliers WHERE supplier_id = 'S01'").iloc[0]
        d = r.json()
        for key in ("supplier_id", "name", "lead_time_days", "on_time_rate"):
            if key not in d or d[key] != truth[key]:
                out.append(f"בגוף התגובה של S01 השדה {key!r} לא תואם למסד")
    if requests.get(BASE2 + "/suppliers/NOPE", timeout=10).status_code != 404:
        out.append("GET /suppliers/NOPE לא החזיר 404")
    if requests.get(BASE2 + "/suppliers/x' OR '1'='1", timeout=10).status_code != 404:
        out.append("השם הזדוני x' OR '1'='1 לא החזיר 404 - ייתכן SQL injection!")
    return out


spec2 = requests.get(BASE2 + "/openapi.json", timeout=10).json()
print("נקודות הקצה בחוזה:", list_endpoints(spec2))
all_problems = None
contract_problems = check_contract(spec2, BASE2)            # הכלי שבניתם בחלק א'
content_problems = supplier_content_problems()
all_problems = contract_problems + content_problems
print()
if all_problems:
    print(f"נמצאו {len(all_problems)} בעיות:")
    for p in all_problems:
        print("  ⚠", p)
else:
    print("✓ לא נמצאו בעיות: הקוד של הכלי עומד בחוזה ובבדיקות התוכן")
print()
show("GET", "/suppliers/S01", base=BASE2)
show("GET", "/suppliers/NOPE", base=BASE2)

In [ ]:
# מלאו את הטבלה (החליפו את הערכים):
ai_log = {
    "tool": "(שם הכלי)",
    "minutes": None,         # כמה דקות לקח עד שהיה קוד שרץ (מספר)
    "problems_found": None,  # כמה בעיות מצאו הכלי שלכם והבדיקות בתא למעלה (מספר)
}

In [ ]:
# בדיקה - ג' שלב 3 (לא תלויה במה שהכלי כתב: בודקת את תיעוד הניסוי ואת *הכלי שלכם*)
def check_c3():
    global get_supplier
    assert ai_log["tool"] != "(שם הכלי)", "ai_log: כתבו את שם הכלי שבו השתמשתם"
    for key in ("minutes", "problems_found"):
        assert isinstance(ai_log[key], (int, float)), f"ai_log: הערך של {key} צריך להיות מספר"
    assert all_problems is not None, "הריצו קודם את תא הבדיקה של הכלי (למעלה)"
    assert ai_log["problems_found"] == len(all_problems), "ai_log: problems_found צריך להתאים למספר הבעיות שהודפסו למעלה"

    def forgetful(supplier_id):                           # באג טיפוסי של AI: שוכח את המקרה שאין שורה
        row = con.execute("SELECT supplier_id, name, lead_time_days, on_time_rate FROM suppliers WHERE supplier_id = ?",
                          (supplier_id,)).fetchone()
        return {"supplier_id": row[0], "name": row[1], "lead_time_days": row[2], "on_time_rate": row[3]}

    def careful(supplier_id):
        row = con.execute("SELECT supplier_id, name, lead_time_days, on_time_rate FROM suppliers WHERE supplier_id = ?",
                          (supplier_id,)).fetchone()
        if row is None:
            raise HTTPException(status_code=404, detail="Supplier not found")
        return {"supplier_id": row[0], "name": row[1], "lead_time_days": row[2], "on_time_rate": row[3]}

    mine = get_supplier
    try:
        get_supplier = forgetful
        assert len(check_contract(spec2, BASE2)) >= 1, "check_contract לא זיהתה נקודת קצה ששוכחת את מקרה ה-404 (היא מחזירה 500, שלא מופיע בחוזה). בדקו שהכלי עובר גם על הכתובת עם NOPE"
        get_supplier = careful
        assert check_contract(spec2, BASE2) == [], "check_contract מצאה בעיות בנקודת קצה תקינה - בדקו את ההשוואה בין קוד הסטטוס (מספר) לקודים בחוזה (טקסט)"
    finally:
        get_supplier = mine
    return True

check_c3() and print("✓ ג' שלב 3: הכלי שלכם מזהה נקודת קצה שמחזירה 500 במקום 404, ויעד הניסוי תועד")

<div dir="rtl">

**(ב)** מה אתם בדקתם ב-AI שלא נראה מהקוד? איזה באג טיפוסי הכלי שלכם תופס (רמז: `forgetful` בתא הבדיקה), ובאיזה קוד הוא מתבטא? ואיזו בדיקה לא הייתה נתפסת בלי הקריאה עם `NOPE`?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ד' - מסקנה (2 דק')

שלושה משפטים, אחד לכל חלק בכיתה. כל משפט בנוי כך: **מה מימשתם ← איזו חוליה/עיקרון זה מראה ← מה עושים בעבודה עם AI.**

1. **HTTP והחוזה:** _(כתבו כאן)_
2. **SQL:** _(כתבו כאן)_
3. **שילוב ו-AI:** _(כתבו כאן)_

</div>

<div dir="rtl">

---
---

# חלקי הבית / ההמשך: ה', ו', ז'

**סיימתם את חלקי החובה (א'-ד').** שלושת החלקים הבאים הם **להמשך או לבית**: ה' (תור) ו-ו' (Docker) הם חלק מהסילבוס ומומלץ מאוד להשלים אותם; ז' אופציונלי. הם עצמאיים (לא תלויים בשרת ובמסד מהחלקים הקודמים).

</div>

<div dir="rtl">

---

# חלק ה' - תור ו-worker: מודל איטי מאחורי תור (12 דק', בית)

## הרקע

מודל שלוקח 10 שניות לא יכול לחסום שרת שמשרת מאה משתמשים. הפתרון: ה-API **לא מריץ** את העבודה האיטית. הוא **רושם משימה בתור, מחזיר מיד "קיבלתי" (קוד 202)**, ו-**worker** (עובד) לוקח משימות מהתור ומבצע אותן ברקע. הלקוח **שואל מדי פעם מה הסטטוס** (*polling*) עד שהמשימה מסתיימת.

**כנות:** ב-Colab אין Redis, Celery ו-Flower, אז נבנה **סימולציה קטנה ואמיתית** של אותו מבנה, בעזרת `queue.Queue` ו-threads (תהליכונים). המבנה זהה:

| בסימולציה כאן | בשירות האמיתי (Celery) |
|---|---|
| `QUEUE` (`queue.Queue`) | **Redis** - התור (broker) |
| `submit_job(text)` | `POST /jobs` ← `classify_email.delay(...)`, מחזיר 202 |
| `worker_loop()` ב-thread | **Celery worker** (`celery -A worker.tasks worker`) |
| `get_status(job_id)` | `GET /jobs/{id}` - `queued`/`running`/`done` |
| `status_counts()` | **Flower** - לוח בקרה: כמה משימות בכל מצב |
| `JOBS` (מילון) | טבלת `jobs` במסד הנתונים |

מחזור החיים של משימה: `PENDING` (בתור) ← `STARTED` (עובד עליה) ← `SUCCESS` (הסתיימה, יש תוצאה).

הריצו את תא ההכנה: הוא מגדיר את התור, את מצב המשימות, ו-`classify(text)` - "המודל" (כללי מילים פשוטים כמו ב-`worker/tasks.py` שבשירות המלא) שלוקח `JOB_DELAY` שניות לכל משימה.

</div>

In [ ]:
QUEUE = queue.Queue()        # התור (בשירות האמיתי: Redis)
JOBS = {}                    # מצב כל משימה (בשירות האמיתי: טבלת jobs במסד)
JOB_DELAY = 0.5              # "המודל האיטי": חצי שנייה לכל משימה
_ids = itertools.count(1)    # מספרים סידוריים למשימות: 1, 2, 3, ...


def _fresh():
    """מאפס את התור והמשימות (לצורך בדיקות והדגמות)."""
    global QUEUE, JOBS, _ids
    QUEUE = queue.Queue()
    JOBS = {}
    _ids = itertools.count(1)


def classify(text):
    """דמוי-מודל: כללי מילים פשוטים (במציאות כאן תהיה קריאה ל-LLM API)."""
    t = text.lower()
    if "invoice" in t:
        return {"category": "invoice", "urgent": False}
    if "late" in t or "delay" in t:
        return {"category": "delay", "urgent": True}
    if "price" in t or "raising" in t:
        return {"category": "price_change", "urgent": False}
    return {"category": "other", "urgent": False}


print("✓ התור מוכן. classify('Shipment will be late') =", classify("Shipment will be late"))

<div dir="rtl">

## שלב 1 - קבלה: `submit_job`

ממשו `submit_job(text)`: זו ה"קבלה" (מה ש-`POST /jobs` עושה). היא **לא מריצה את המודל**: היא רק (1) מקצה מספר משימה, (2) רושמת את המשימה ב-`JOBS` עם סטטוס `"PENDING"` ותוצאה `None`, (3) מכניסה את מספר המשימה ל-`QUEUE`, ו-(4) **מחזירה את המספר מיד**.

**כלים:** `job_id = next(_ids)` נותן את המספר הסידורי הבא. `JOBS[job_id] = {"text": ..., "status": ..., "result": ...}` רושם. `QUEUE.put(job_id)` מכניס לתור.

</div>

In [ ]:
def submit_job(text):
    """Registers a job and puts it in the queue; does NOT run the model. Returns the job_id (an int) immediately.
    JOBS[job_id] = {"text": text, "status": "PENDING", "result": None}; QUEUE.put(job_id)
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ה' שלב 1
def check_e1():
    _fresh()
    t0 = time.time()
    j1 = submit_job("Shipment will be late")
    j2 = submit_job("invoice attached")
    elapsed = time.time() - t0
    assert isinstance(j1, int) and j1 != j2, "submit_job: צריכה להחזיר מספר משימה (int), שונה בכל קריאה"
    assert elapsed < JOB_DELAY, "submit_job: חייבת לחזור מיד (כמו 202) - בלי להריץ את המודל ובלי sleep. את העבודה האיטית עושה ה-worker"
    assert JOBS[j1]["status"] == "PENDING" and JOBS[j1]["result"] is None, "submit_job: במשימה חדשה status הוא 'PENDING' ו-result הוא None"
    assert JOBS[j1]["text"] == "Shipment will be late", "submit_job: שמרו ב-JOBS גם את הטקסט, בשדה 'text'"
    assert QUEUE.qsize() == 2, "submit_job: צריך להכניס את מספר המשימה לתור (QUEUE.put)"
    assert QUEUE.get_nowait() == j1, "submit_job: לתור נכנס מספר המשימה (job_id)"
    return True

check_e1() and print("✓ ה' שלב 1: submit_job רושמת ומחזירה מיד")

<div dir="rtl">

## שלב 2 - העובד: `worker_loop`

ממשו `worker_loop()`: זו הלולאה של ה-worker, שרצה ב-thread נפרד. בכל סיבוב היא:
1. לוקחת מספר משימה מהתור: `job_id = QUEUE.get()` (**מחכה** עד שיש משהו);
2. אם קיבלה `None` - זה "אות עצירה": **יוצאת מהלולאה** (`break`);
3. מסמנת `status = "STARTED"`;
4. "מריצה את המודל": `time.sleep(JOB_DELAY)` ואז `result = classify(text)`;
5. שומרת את התוצאה **ואז** מסמנת `status = "SUCCESS"` (בסדר הזה - כדי שמי שרואה SUCCESS ימצא תוצאה).

**כלים:** `while True:` ו-`break`.

</div>

In [ ]:
def worker_loop():
    """Runs forever in its own thread. Each round: take a job_id from QUEUE (blocks until there is one);
    if it is None -> stop (break); else set status "STARTED", sleep(JOB_DELAY) (the slow model),
    store result = classify(text), and finally set status "SUCCESS".
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ה' שלב 2
def check_e2():
    global JOB_DELAY
    _fresh()
    old_delay, JOB_DELAY = JOB_DELAY, 0.2
    texts = ["Shipment will be late", "Thanks, invoice attached", "hello there"]
    t = None
    try:
        ids = [submit_job(x) for x in texts]
        QUEUE.put(None)                                  # אות עצירה אחרי שלוש המשימות
        errors = []
        def runner():
            try:
                worker_loop()
            except Exception as e:
                errors.append(e)
        t = threading.Thread(target=runner, daemon=True)
        t.start()
        seen, deadline = set(), time.time() + 6
        while t.is_alive() and time.time() < deadline:
            seen.add(JOBS[ids[0]]["status"])
            time.sleep(0.01)
        assert not errors, f"worker_loop נכשלה עם שגיאה ({errors[0]!r}). האם טיפלתם ב-None (אות עצירה) לפני שניגשים ל-JOBS?"
        assert not t.is_alive(), "worker_loop: צריכה לצאת מהלולאה (break) כשהיא מקבלת None מהתור"
        assert "STARTED" in seen, "worker_loop: צריך לסמן status='STARTED' בזמן שהמשימה מתבצעת (לפני ה-sleep)"
        for job_id, text in zip(ids, texts):
            assert JOBS[job_id]["status"] == "SUCCESS", f"worker_loop: משימה {job_id} צריכה להסתיים ב-'SUCCESS', התקבל {JOBS[job_id]['status']!r}"
            assert JOBS[job_id]["result"] == classify(text), f"worker_loop: result של משימה {job_id} צריך להיות classify(text)"
    finally:
        JOB_DELAY = old_delay
        if t is not None and t.is_alive():
            QUEUE.put(None)
            t.join(1)
    return True

check_e2() and print("✓ ה' שלב 2: ה-worker לוקח מהתור, עובד ומסיים")

<div dir="rtl">

## שלב 3 - סטטוס: `get_status`

ממשו `get_status(job_id)`: זה מה ש-`GET /jobs/{id}` מחזיר ללקוח ששואל "מה קורה עם המשימה שלי?". מחזירה מילון `{"job_id": ..., "status": ..., "result": ...}`, ו-**`None` אם אין משימה כזאת** (ב-API האמיתי: 404).

**כלי:** `JOBS.get(job_id)` מחזיר `None` אם אין מפתח כזה.

</div>

In [ ]:
def get_status(job_id):
    """Returns {"job_id": job_id, "status": ..., "result": ...} for a known job, or None if there is no such job.
    e.g. get_status(1) -> {"job_id": 1, "status": "PENDING", "result": None}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ה' שלב 3
def check_e3():
    _fresh()
    JOBS[7] = {"text": "x", "status": "STARTED", "result": None}
    JOBS[8] = {"text": "y", "status": "SUCCESS", "result": {"category": "other", "urgent": False}}
    assert get_status(7) == {"job_id": 7, "status": "STARTED", "result": None}, "get_status: צריך להחזיר {'job_id':..., 'status':..., 'result':...} (בלי 'text')"
    assert get_status(8) == {"job_id": 8, "status": "SUCCESS", "result": {"category": "other", "urgent": False}}, "get_status: על משימה שהסתיימה צריך להחזיר גם את ה-result"
    assert get_status(99) is None, "get_status: על משימה שלא קיימת צריך להחזיר None (ב-API: 404)"
    return True

check_e3() and print("✓ ה' שלב 3: get_status")

<div dir="rtl">

## ניסוי 1 - העובד כבוי: המשימה לא אובדת

זה הרגע שמראה **מה נותן תור**. נשלח שלוש בקשות **כשאין worker**, ואחר כך נפעיל אחד.

</div>

<div dir="rtl">

**לפני שמריצים:** שולחים 3 בקשות כשאין עובד. כמה זמן ייקח לכל `submit_job` (מיידי, או חצי שנייה כמו המודל)? מה יהיה הסטטוס שלהן? והאם הן יאבדו? ומה יקרה אחרי שנפעיל עובד?

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
_fresh()
texts = ["Shipment will be late", "Thanks for your order, invoice attached.", "We are raising prices by 8% next month."]
for text in texts:
    t0 = time.time()
    job_id = submit_job(text)
    print(f"submit_job -> {job_id}   ({(time.time() - t0) * 1000:.1f} מילישניות)   סטטוס: {get_status(job_id)['status']}")
print("בתור כרגע:", QUEUE.qsize(), "משימות. אין worker, והמשימות לא אבדו.")
ids = list(JOBS)

In [ ]:
worker = threading.Thread(target=worker_loop, daemon=True)
worker.start()                                  # מפעילים עובד
print("worker הופעל. polling - שואלים כל רבע שנייה:")
t0 = time.time()
while any(get_status(i)["status"] != "SUCCESS" for i in ids):
    if time.time() - t0 > 20:
        raise TimeoutError("המשימות לא הסתיימו - בדקו את worker_loop")
    print(f"  t={time.time() - t0:.2f}s", [get_status(i)["status"] for i in ids])
    time.sleep(0.25)
print(f"  t={time.time() - t0:.2f}s", [get_status(i)["status"] for i in ids])
for i in ids:
    print(i, get_status(i)["result"])
QUEUE.put(None); worker.join()                  # אות עצירה

<div dir="rtl">

## ניסוי 2 - מוסיפים עובדים

מה קורה כשיש הרבה בקשות? **מוסיפים עובדים, בלי לגעת ב-API.** `run_batch(n_jobs, n_workers)` (כבר מוגדרת) מפעילה `n_workers` עובדים, שולחת `n_jobs` משימות של חצי שנייה ומחזירה בכמה שניות הכול הסתיים.

</div>

In [ ]:
def run_batch(n_jobs, n_workers):
    """מפעילה n_workers עובדים, שולחת n_jobs משימות, ומחכה שכולן יסתיימו. מחזירה כמה שניות זה לקח."""
    _fresh()
    workers = [threading.Thread(target=worker_loop, daemon=True) for _ in range(n_workers)]
    for w in workers:
        w.start()
    t0 = time.time()
    ids = [submit_job(f"email number {i}") for i in range(n_jobs)]
    while any(get_status(i)["status"] != "SUCCESS" for i in ids):
        if time.time() - t0 > 30:
            raise TimeoutError("המשימות לא הסתיימו - בדקו את worker_loop")
        time.sleep(0.02)
    elapsed = time.time() - t0
    for _ in workers:
        QUEUE.put(None)
    for w in workers:
        w.join()
    return elapsed

print("✓ run_batch מוגדרת")

<div dir="rtl">

**לפני שמריצים:** 6 משימות, כל אחת חצי שנייה. בכמה שניות יסתיים הכול עם עובד אחד? עם שניים? עם שלושה? (חישוב: כמה סיבובים צריך כל עובד?)

**התחזית שלי:** _(כתבו כאן)_

</div>

In [ ]:
for n_workers in (1, 2, 3):
    print(f"עובדים: {n_workers}  ←  {run_batch(6, n_workers):.2f} שניות")

<div dir="rtl">

## שלב 4 - לוח בקרה בקטן: `status_counts` (Flower)

**Flower** הוא מסך שמראה כמה משימות יש בכל מצב. בסימולציה: ממשו `status_counts()` שמחזירה מילון **כמה משימות יש בכל סטטוס** (רק סטטוסים שמופיעים), למשל `{"SUCCESS": 2, "PENDING": 1}`.

**כלי:** `Counter(job["status"] for job in JOBS.values())` סופר כמה פעמים כל ערך מופיע. (`dict(...)` הופך אותו למילון רגיל.)

</div>

In [ ]:
def status_counts():
    """Returns a dict: status -> how many jobs have it (only statuses that appear).
    e.g. {"SUCCESS": 2, "PENDING": 1}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ה' שלב 4
def check_e4():
    _fresh()
    for status in ("SUCCESS", "SUCCESS", "PENDING", "STARTED", "SUCCESS"):
        JOBS[len(JOBS) + 1] = {"text": "x", "status": status, "result": None}
    assert status_counts() == {"SUCCESS": 3, "PENDING": 1, "STARTED": 1}, f"status_counts: ציפינו ל-{{'SUCCESS': 3, 'PENDING': 1, 'STARTED': 1}}, התקבל {status_counts()}"
    _fresh()
    assert status_counts() == {}, "status_counts: כשאין משימות צריך להחזיר מילון ריק"
    return True

check_e4() and print("✓ ה' שלב 4: status_counts")

In [ ]:
_fresh()
for i in range(4):
    submit_job(f"email {i}")
print("לפני שהעובד התחיל:", status_counts())
w = threading.Thread(target=worker_loop, daemon=True); w.start()
time.sleep(JOB_DELAY * 1.5)
print("באמצע:           ", status_counts())
QUEUE.put(None); w.join()
print("בסוף:            ", status_counts())

<div dir="rtl">

**(א)** מדוע `submit_job` חוזרת מיד, ומה הקשר לקוד 202 ולמשפט "מודל שלוקח 10 שניות לא יכול לחסום שרת שמשרת מאה משתמשים"? מה היה קורה אילו `submit_job` הייתה מריצה את `classify` בעצמה ו-10 משתמשים שולחים בו-זמנית?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ב)** בניסוי 1 העובד היה כבוי והמשימות לא אבדו. ומה **עוד** חסר כדי שזה יעבוד בשירות אמיתי - איזו בעיה יש אם העובד נופל *באמצע* משימה (סטטוס `STARTED` לנצח)? ומה בא במקום זה ב-Celery/Flower? (רמז: שקף "מחזור החיים של משימה")

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ג)** מי הוא התור ומי הספרייה בעולם האמיתי: Redis, Celery, Flower - מה התפקיד של כל אחד, וכנגד מה בסימולציה שלכם הוא עומד? ומה הקשר בין "הוספת עובדים בניסוי 2" לבין "ה-API נשאר דק"?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**מסקנה לחלק ה'** (משפט אחד, בנוי: *מה מימשתי ← איזו חוליה זה מראה ← מה עושים בעבודה עם AI*): _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ו' - Docker ו-Kubernetes: קוראים, בודקים ומבינים (12 דק', בית)

## הרקע

"**אצלי זה עובד**" - אבל אצל חבר, ובשרת, לא. **Docker** פותר את זה בכך שהוא **אורז את הקוד יחד עם הסביבה** (Python, ספריות, הגדרות). שלושה מושגים: **Dockerfile** = המתכון; **image** = תבנית שנבנית ממנו (קריאה בלבד); **container** = מופע שרץ מהתמונה. **קונטיינר אינו מכונה וירטואלית:**

| | מכונה וירטואלית (VM) | קונטיינר (Docker) |
|---|---|---|
| מה אורזים | מערכת הפעלה **שלמה** לכל אפליקציה, מעל Hypervisor | רק את האפליקציה והספריות שלה; **גרעין המארח משותף** |
| גודל והפעלה | גיגה-בייטים, דקות | עשרות עד מאות מגה, שניות |
| בידוד | חזק | קל יותר |

**docker compose** מפעיל כמה קונטיינרים יחד בפקודה אחת (`docker compose up`). **Kubernetes (k8s)** מנהל **הרבה** קונטיינרים על **הרבה** מכונות: אתם מצהירים "אני רוצה 3 עותקים", והוא דואג לכך (ומחליף עותק שנפל). בפרויקט שלכם: compose מספיק, ו-k8s להכרה בלבד.

**כנות:** ב-Colab אי אפשר להריץ Docker או Kubernetes. מה שכן אפשר, וזה מה ש**כלי AI ועמיתים עושים בפועל** - לקרוא קבצי תצורה כקוד ולבדוק אותם (בעולם האמיתי קיימים כלי `lint` ל-Dockerfile). כאן הקבצים הם **טקסט**, ואתם תכתבו כלים שמנתחים אותם. ההרצה האמיתית של Docker ו-compose - במעבדות של המפגש או בפרויקט, על המחשב שלכם.

הקבצים (מהחבילה שבמפגש): ה-`Dockerfile`, `docker-compose.yml` ו-`k8s/deployment.yaml` של שירות החנות. קראו אותם:

</div>

In [ ]:
DOCKERFILE = """# One image for the API, the worker and the UI: the command decides what runs.
FROM python:3.11-slim
WORKDIR /srv
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

COMPOSE = """# Five containers that work together: queue (redis), api, worker, monitor (flower) and ui.
services:
  redis:
    image: redis:7-alpine

  api:
    build: .
    ports: ["8000:8000"]
    environment:
      CELERY_BROKER_URL: redis://redis:6379/0
      DB_PATH: /data/store.db
    volumes: ["dbdata:/data"]
    depends_on: [redis]

  worker:
    build: .
    command: celery -A worker.tasks worker --loglevel=info
    environment:
      CELERY_BROKER_URL: redis://redis:6379/0
      DB_PATH: /data/store.db
    volumes: ["dbdata:/data"]
    depends_on: [redis]

  flower:
    build: .
    command: celery -A worker.tasks flower --port=5555
    ports: ["5555:5555"]
    environment:
      CELERY_BROKER_URL: redis://redis:6379/0
    depends_on: [redis]

  ui:
    build: .
    command: streamlit run ui/app.py --server.port 8501 --server.address 0.0.0.0
    ports: ["8501:8501"]
    environment:
      API_URL: http://api:8000
    depends_on: [api]

volumes:
  dbdata:
"""

K8S = """# Illustration only: what the API would look like on Kubernetes.
apiVersion: apps/v1
kind: Deployment
metadata:
  name: store-api
spec:
  replicas: 3                      # run 3 identical copies (pods)
  selector:
    matchLabels: {app: store-api}
  template:
    metadata:
      labels: {app: store-api}
    spec:
      containers:
        - name: api
          image: myregistry/store-service:1.0.0
          ports:
            - containerPort: 8000
          env:
            - name: CELERY_BROKER_URL
              value: redis://redis:6379/0
---
apiVersion: v1
kind: Service
metadata:
  name: store-api
spec:
  selector: {app: store-api}       # a stable address in front of the 3 pods
  ports:
    - port: 80
      targetPort: 8000
"""

print(DOCKERFILE)

<div dir="rtl">

## שלב 1 - קורא Dockerfile: `parse_dockerfile`

ממשו `parse_dockerfile(text)`: מחזירה **רשימה של tuple** `(INSTRUCTION, ארגומנטים)` לפי הסדר - שורה אחת לכל הוראה, ההוראה באותיות גדולות. **מדלגים** על שורות ריקות ועל הערות (שורות שמתחילות ב-`#`). (נניח שכל הוראה בשורה אחת, בלי שורות המשך.)

**כלים:** `text.splitlines()`, `line.strip()`, `line.startswith("#")`. `line.split(None, 1)` מפצל **פעם אחת בלבד** לפי רווח: `"FROM python:3.11-slim".split(None, 1)` נותן `["FROM", "python:3.11-slim"]`.

</div>

In [ ]:
def parse_dockerfile(text):
    """Returns a list of (INSTRUCTION, arguments) tuples in order (INSTRUCTION in capitals).
    Skips empty lines and lines starting with #. Assume every instruction is on ONE line.
    e.g. "# hi\nfrom python:3.11-slim\nWORKDIR /srv" -> [("FROM", "python:3.11-slim"), ("WORKDIR", "/srv")]
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ו' שלב 1
def check_f1():
    got = [tuple(x) for x in parse_dockerfile("# a comment\n\nfrom python:3.11-slim\n  WORKDIR /srv\nCMD [\"uvicorn\", \"app:app\"]\n")]
    assert got == [("FROM", "python:3.11-slim"), ("WORKDIR", "/srv"), ("CMD", '["uvicorn", "app:app"]')], \
        f"parse_dockerfile: ציפינו ל-3 הוראות (בלי ההערה והשורה הריקה, ההוראה באותיות גדולות), התקבל {got}"
    real = [tuple(x) for x in parse_dockerfile(DOCKERFILE)]
    assert [name for name, _ in real] == ["FROM", "WORKDIR", "COPY", "RUN", "COPY", "EXPOSE", "CMD"], \
        f"parse_dockerfile: ב-Dockerfile של השירות 7 הוראות: FROM, WORKDIR, COPY, RUN, COPY, EXPOSE, CMD. התקבל {[n for n, _ in real]}"
    assert real[0] == ("FROM", "python:3.11-slim"), "parse_dockerfile: ההוראה הראשונה צריכה להיות ('FROM', 'python:3.11-slim')"
    return True

check_f1() and print("✓ ו' שלב 1:", [name for name, _ in parse_dockerfile(DOCKERFILE)])

<div dir="rtl">

## שלב 2 - בודק Dockerfile: שלושה כללים

עכשיו נבדוק Dockerfile **מול כללים**, כמו שבודקים קוד. כל כלל הוא פונקציה קצרה שמקבלת את רשימת ההוראות (`cmds` - התוצאה של `parse_dockerfile`) ומחזירה **רשימת בעיות** (טקסטים; ריקה = בסדר). **כל כלל שמופר מוסיף בעיה אחת.**

**כלל 1 - תמונת בסיס מוצמדת:** `rule_base_image(cmds)`. בכל `FROM`: אם לתמונה **אין tag** (אין `:` בשם, כמו `python`) **או** שה-tag הוא `latest` (`python:latest`) - בעיה. סיבה: `latest` משתנה עם הזמן, והבנייה של מחר לא תהיה כמו של היום. (שם התמונה הוא המילה הראשונה בארגומנטים: `args.split()[0]`.)

**כלל 2 - requirements לפני הקוד:** `rule_copy_order(cmds)`. אם יש `COPY` של הקוד (ארגומנטים שמתחילים ב-`". "`, כמו `COPY . .`) **לפני** שהיה `COPY` של `requirements` (ארגומנטים שמתחילים ב-`requirements`) - בעיה. סיבה: כל שורה היא שכבה (layer) שנשמרת במטמון; אם מעתיקים את כל הקוד לפני ההתקנה, כל שינוי קטן בקוד מאלץ התקנה מחדש של כל הספריות.

**כלל 3 - בלי סיסמה ב-`ENV`:** `rule_no_secrets(cmds)`. הוראת `ENV` שהארגומנטים שלה מכילים (באותיות גדולות) `PASSWORD`, `SECRET`, `TOKEN` או `API_KEY` - בעיה. סיבה: ההוראה נשמרת בתוך ה-image, וכל מי שמקבל אותו רואה אותה (סודות עוברים בזמן הרצה, מחוץ ל-image: מפגש 10).

**כלים:** `for name, args in cmds:`; `name == "FROM"`; `args.upper()`; `any(word in args.upper() for word in (...))`.

</div>

In [ ]:
def rule_base_image(cmds):
    """cmds: list of (INSTRUCTION, arguments). For every FROM whose image has no tag (no ":") or ends with ":latest"
    add one problem text. Returns the list of problems.
    e.g. [("FROM", "python")] -> one problem; [("FROM", "python:3.11-slim")] -> []
    """
    problems = []
    # TODO
    raise NotImplementedError


def rule_copy_order(cmds):
    """cmds: list of (INSTRUCTION, arguments). A COPY whose arguments start with ". " (the code) that comes BEFORE
    a COPY whose arguments start with "requirements" -> one problem text (also when there is no requirements COPY at all).
    Returns the list of problems (at most one).
    """
    problems = []
    # TODO
    raise NotImplementedError


def rule_no_secrets(cmds):
    """cmds: list of (INSTRUCTION, arguments). Every ENV instruction whose arguments (upper-cased) contain
    PASSWORD, SECRET, TOKEN or API_KEY -> one problem text. Returns the list of problems.
    e.g. [("ENV", "DB_PASSWORD=1234")] -> one problem
    """
    problems = []
    # TODO
    raise NotImplementedError


def lint_dockerfile(text):
    """בודק Dockerfile מול שלושת הכללים. מחזיר רשימת בעיות (ריקה = בסדר)."""
    cmds = parse_dockerfile(text)
    return rule_base_image(cmds) + rule_copy_order(cmds) + rule_no_secrets(cmds)

In [ ]:
# בדיקה - ו' שלב 2
def check_f2():
    ok = [("FROM", "python:3.11-slim"), ("COPY", "requirements.txt ."), ("RUN", "pip install -r requirements.txt"), ("COPY", ". .")]
    assert rule_base_image(ok) == [], "rule_base_image: python:3.11-slim מוצמדת ל-tag - אין בעיה"
    assert len(rule_base_image([("FROM", "python")])) == 1, "rule_base_image: FROM python (בלי tag) - בעיה אחת"
    assert len(rule_base_image([("FROM", "python:latest")])) == 1, "rule_base_image: FROM python:latest - בעיה אחת"
    assert len(rule_base_image([("FROM", "python:3.11 AS build"), ("FROM", "node")])) == 1, "rule_base_image: רק ה-FROM השני (node, בלי tag) בעייתי; 'AS build' לא משנה את שם התמונה"
    assert rule_copy_order(ok) == [], "rule_copy_order: requirements לפני הקוד - אין בעיה"
    bad_order = [("FROM", "python:3.11"), ("COPY", ". ."), ("RUN", "pip install -r requirements.txt")]
    assert len(rule_copy_order(bad_order)) == 1, "rule_copy_order: COPY . . לפני התקנת requirements - בעיה אחת"
    assert len(rule_copy_order([("FROM", "python:3.11"), ("COPY", ". ."), ("COPY", "requirements.txt .")])) == 1, "rule_copy_order: גם אם requirements מועתק *אחרי* הקוד - בעיה"
    assert rule_no_secrets([("ENV", "APP_MODE=prod")]) == [], "rule_no_secrets: APP_MODE אינו סוד"
    assert len(rule_no_secrets([("ENV", "DB_PASSWORD=1234"), ("ENV", "api_key=abc")])) == 2, "rule_no_secrets: DB_PASSWORD ו-api_key (גם באותיות קטנות) - שתי בעיות"
    assert lint_dockerfile(DOCKERFILE) == [], f"lint_dockerfile: ה-Dockerfile של השירות תקין, אבל התקבלו בעיות: {lint_dockerfile(DOCKERFILE)}"
    return True

check_f2() and print("✓ ו' שלב 2: שלושת הכללים עובדים, וה-Dockerfile של השירות נקי")

In [ ]:
BAD_DOCKERFILE = """
FROM python
COPY . .
RUN pip install -r requirements.txt
ENV DB_PASSWORD=1234
CMD ["python", "main.py"]
"""
for problem in lint_dockerfile(BAD_DOCKERFILE):
    print("⚠", problem)

<div dir="rtl">

## שלב 3 - קורא compose: `compose_services`

הקובץ `docker-compose.yml` הוא **YAML** (טקסט מובנה כמו JSON). הספרייה `yaml` (מותקנת ב-Colab) הופכת אותו למילון: `yaml.safe_load(text)`. ממשו `compose_services(yaml_text)`: מחזירה מילון **שם שירות ← רשימת השירותים שהוא תלוי בהם** (`depends_on`), ורשימה ריקה לשירות בלי תלות. כך רואים את **תלויות** המערכת.

**כלים:** `data["services"]` הוא מילון של שירותים. `svc.get("depends_on", [])` מחזיר את התלויות (או רשימה ריקה). בקובץ compose `depends_on` יכול להיות **רשימה** או **מילון** - `list(...)` של כל אחד מהם נותן רשימת שמות.

</div>

In [ ]:
def compose_services(yaml_text):
    """Returns a dict: service name -> list of the service names it depends on (depends_on), [] if none.
    depends_on may be a list or a dict (then the names are its keys).
    e.g. "services:\n  db: {image: postgres}\n  app: {build: ., depends_on: [db]}" -> {"db": [], "app": ["db"]}
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ו' שלב 3
def check_f3():
    got = compose_services(COMPOSE)
    assert got == {"redis": [], "api": ["redis"], "worker": ["redis"], "flower": ["redis"], "ui": ["api"]}, \
        f"compose_services: על קובץ השירות ציפינו ל-{{'redis': [], 'api': ['redis'], 'worker': ['redis'], 'flower': ['redis'], 'ui': ['api']}}, התקבל {got}"
    long_form = "services:\n  db:\n    image: postgres:16\n  app:\n    build: .\n    depends_on:\n      db:\n        condition: service_healthy\n"
    assert compose_services(long_form) == {"db": [], "app": ["db"]}, "compose_services: כש-depends_on הוא מילון, התלויות הן המפתחות (list(...) של מילון)"
    return True

check_f3() and print("✓ ו' שלב 3:", compose_services(COMPOSE))

<div dir="rtl">

## שלב 4 - Kubernetes: מה מצהירים? `k8s_replicas`

ב-Kubernetes **מצהירים** מה רוצים (קובץ YAML), והמערכת דואגת לכך. הקובץ `deployment.yaml` מכיל **כמה מסמכים** (מופרדים ב-`---`): `Deployment` (כמה עותקים - `replicas`) ו-`Service` (כתובת קבועה מול העותקים). ממשו `k8s_replicas(yaml_text)`: מחזירה את מספר ה-`replicas` של ה-`Deployment`, או `None` אם אין Deployment בקובץ.

**כלים:** `yaml.safe_load_all(text)` עובר על כל המסמכים (`for doc in yaml.safe_load_all(text):`). `doc["kind"]` אומר איזה סוג, ו-`doc["spec"]["replicas"]` הוא המספר.

</div>

In [ ]:
def k8s_replicas(yaml_text):
    """yaml_text may hold several YAML documents. Returns spec.replicas of the document whose kind is "Deployment",
    or None if there is no Deployment.
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ו' שלב 4
def check_f4():
    assert k8s_replicas(K8S) == 3, f"k8s_replicas: ב-deployment.yaml של השירות יש replicas: 3, התקבל {k8s_replicas(K8S)}"
    only_service = "apiVersion: v1\nkind: Service\nmetadata:\n  name: x\nspec:\n  ports:\n    - port: 80\n"
    assert k8s_replicas(only_service) is None, "k8s_replicas: בלי Deployment צריך להחזיר None"
    five = "kind: Service\n---\nkind: Deployment\nspec:\n  replicas: 5\n"
    assert k8s_replicas(five) == 5, "k8s_replicas: צריך לחפש את ה-Deployment בכל המסמכים, גם כשהוא לא הראשון"
    return True

check_f4() and print("✓ ו' שלב 4: replicas =", k8s_replicas(K8S))

<div dir="rtl">

**(א)** הכלל "`COPY requirements.txt` לפני `COPY . .`" - מה קורה בבנייה מחדש (`docker build`) אחרי ששינינו שורה אחת בקוד, עם הסדר הנכון ועם הסדר ההפוך? (רמז: שכבות ומטמון)

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ב)** מה ההבדל בין **image** ל-**container**, ובין **קונטיינר** ל-**מכונה וירטואלית (VM)**? ומה המשמעות של ההבדל (מה מרוויחים ומה מפסידים)?

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**(ג)** לפי הפלט של `compose_services`: אם `redis` נופל, אילו שירותים מושפעים ישירות? ומה בקובץ ה-compose שלנו שונה בעקרון מהרצת ארבעה טרמינלים ידנית? ומה **Kubernetes** נותן שקבצי compose על מחשב אחד לא נותנים? (רמז: `k8s_replicas`)

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**מסקנה לחלק ו'** (משפט אחד: *מה מימשתי ← איזו חוליה זה מראה ← מה עושים בעבודה עם AI*): _(כתבו כאן)_

</div>

<div dir="rtl">

---

# חלק ז' - קריאה ל-API חיצוני (אופציונלי, 6 דק', בית)

## הרקע

פרויקט הגמר דורש קריאה ל-API של צד שלישי. כל קריאה כזו יכולה להיכשל בדרכים שאתם לא שולטים בהן: **השרת איטי** (בלי `timeout` הקוד נתקע), **מגבלת קצב** (429), **תקלה אצלם** (5xx), או **פרמטר שהמצאתם** (השרת מתעלם בשקט ומחזיר 200 עם נתונים שגויים - ההזיה המסוכנת ביותר, שקף "הזיות API"). לכן כותבים את הקריאה **בזהירות**, ובודקים אותה **בלי** תלות בשירות החי: מזריקים לפונקציה "שרת מזויף" שמחזיר את התגובה שרוצים לבדוק.

התיעוד של שירות שער החליפין (Frankfurter, פתוח וללא מפתח): כתובת `https://api.frankfurter.dev/v1/latest` (גרסה 1; באתר היא מוגדרת כ"ישנה אך זמינה ללא הגבלת זמן", ובפרויקט כדאי לבדוק בתיעוד אם יש גרסה עדכנית יותר), **שני פרמטרים בלבד**: `base` (מטבע מקור) ו-`symbols` (מטבע יעד). תגובה מוצלחת: `{"amount": 1.0, "base": "USD", "date": "2026-10-16", "rates": {"ILS": 3.0945}}` - **הערך נמצא בתוך `rates`**.

ממשו `get_rate(base, symbol, http_get=requests.get)`: מחזירה את השער (מספר), או **`None`** אם משהו השתבש: חריגת רשת (`requests.RequestException`, למשל `Timeout`) או קוד סטטוס שאינו 200. חובה `timeout=10`, ופרמטרים `params={"base": ..., "symbols": ...}` בלבד.

**כלים:** `try: ... except requests.RequestException: return None`. `r = http_get(url, params=..., timeout=10)`, `r.status_code`, `r.json()["rates"][symbol]`. הפרמטר `http_get` מאפשר לבדיקה להחליף את `requests.get` בשרת מזויף.

</div>

In [ ]:
def get_rate(base, symbol, http_get=requests.get):
    """Gets the exchange rate base -> symbol from https://api.frankfurter.dev/v1/latest (params: base, symbols; timeout=10).
    Returns the rate (float) = response.json()["rates"][symbol], or None on a network error
    (requests.RequestException) or when the status code is not 200.
    e.g. get_rate("USD", "ILS") -> 3.0945
    """
    # TODO
    raise NotImplementedError

In [ ]:
# בדיקה - ז' (בלי אינטרנט: משתמשים בשרת מזויף)
class _Fake:
    def __init__(self, code, data=None):
        self.status_code, self._data = code, data
    def json(self):
        return self._data

def check_g():
    calls = []
    def good(url, **kw):
        calls.append((url, kw))
        return _Fake(200, {"amount": 1.0, "base": "USD", "date": "2026-10-16", "rates": {"ILS": 3.0945}})
    assert get_rate("USD", "ILS", http_get=good) == 3.0945, "get_rate: על תגובה תקינה צריך להחזיר את הערך מתוך ['rates']['ILS']"
    url, kw = calls[0]
    assert "frankfurter" in url, "get_rate: הכתובת צריכה להיות של Frankfurter (api.frankfurter.dev/v1/latest)"
    assert "timeout" in kw, "get_rate: חובה timeout - בלעדיו הקוד עלול להיתקע לנצח"
    assert set(kw.get("params", {})) <= {"base", "symbols"}, f"get_rate: הפרמטרים המתועדים הם base ו-symbols בלבד; נשלחו {set(kw.get('params', {}))} (פרמטר מומצא = השרת יתעלם בשקט!)"
    assert kw["params"].get("base") == "USD" and kw["params"].get("symbols") == "ILS", "get_rate: params צריך להיות {'base': 'USD', 'symbols': 'ILS'}"
    for code in (404, 429, 500):
        assert get_rate("USD", "ILS", http_get=lambda url, code=code, **kw: _Fake(code, {})) is None, f"get_rate: על קוד {code} צריך להחזיר None (לא לקרוס ולא להחזיר נתונים)"
    def slow(url, **kw):
        raise requests.Timeout("too slow")
    assert get_rate("USD", "ILS", http_get=slow) is None, "get_rate: על Timeout צריך להחזיר None (תפסו requests.RequestException)"
    return True

check_g() and print("✓ ז': get_rate מטפלת ב-200, 404, 429, 500 ובתקלת רשת")

In [ ]:
# קריאה חיה (דורשת אינטרנט; לא חלק מהבדיקה). אם אין תשובה - משתמשים בדוגמה השמורה
CACHED = {"amount": 1.0, "base": "USD", "date": "2026-10-16", "rates": {"ILS": 3.0945}}
try:
    rate = get_rate("USD", "ILS")
except NotImplementedError:
    rate = None
print("שער חי USD -> ILS:", rate if rate else f"לא התקבל. דוגמה שמורה מ-{CACHED['date']}: {CACHED['rates']['ILS']}")

<div dir="rtl">

**(א)** מה צריכה הפונקציה לעשות על **429** ומה על **500**? האם מנסים שוב באותה צורה? ולמה כתבנו את הבדיקה עם שרת **מזויף** ולא מול השירות האמיתי? (רמז: שקף "קודי סטטוס", "מפתח וקצב")

**תשובה:** _(כתבו כאן)_

</div>

<div dir="rtl">

**מסקנה לחלק ז'** (משפט אחד): _(כתבו כאן)_

</div>

<div dir="rtl">

---

# בונוס

**1. גבולות מתוך החוזה.** כתבו `limit_bounds(spec)` שמחזירה `(minimum, maximum)` של הפרמטר `limit` ב-`GET /products` **מתוך `openapi.json`** (רמז: הדפסה בשלב 4), ואז בדיקות גבול: `limit=minimum-1` ← 422, `limit=maximum` ← 200, `limit=maximum+1` ← 422. כך הבדיקה נגזרת מהחוזה ולא מהקוד: אם מישהו ישנה את הגבול, הבדיקה תתעדכן.

**2. SQL נוסף.** כתבו שאילתה שמחזירה **כמה הזמנות יש בכל חודש** (רמז: `SUBSTR(order_date, 1, 7)` נותן `2025-01`, ואז `GROUP BY`). איך תוודאו שהתוצאה נכונה? (רמז: סכום ההזמנות בכל החודשים).

**3. מבנה מלא.** חברו את חלק ה' ל-API: בנו אפליקציית FastAPI שלישית (על `start_server`) עם `POST /jobs` שמחזירה **202** ו-`GET /jobs/{id}` שמחזירה 404 ל-id לא קיים - בעזרת `submit_job` ו-`get_status`. הפעילו עובד, והריצו `check_contract` עליה.

</div>

In [ ]:
# תא לבונוס 1

In [ ]:
# תא לבונוס 2

In [ ]:
# תא לבונוס 3

<div dir="rtl">

---

# בדיקה מסכמת - הריצו לפני ההגשה

התא הבא מריץ שוב את כל תאי הבדיקה. לפני ההגשה: **Runtime ← Restart session and run all**, וודאו שכל השורות מסומנות `✓` (בחלקים שהגעתם אליהם - חלקי הבית אינם חובה בכיתה).

</div>

In [ ]:
groups = [
    ("חלקי חובה (בכיתה)", [
        ("א' - status_family", check_a1),
        ("א' - find_product (404)", check_a2),
        ("א' - create_order (201/422/404)", check_a3),
        ("א' - list_endpoints", check_a5),
        ("א' - check_contract", check_a6),
        ("ב' - q_slow_suppliers", check_b1),
        ("ב' - q_category_revenue (JOIN)", check_b2),
        ("ב' - q_top_products (?)", check_b3),
        ("ב' - join_multiplies", check_b4),
        ("ב' - find_products_safe", check_b5),
        ("ג' - top_products_from_db", check_c1),
        ("ג' - e2e_test (+ 4 גרסאות שגויות)", check_c2),
        ("ג' - בדיקת ה-AI והכלי שלכם", check_c3),
    ]),
    ("חלקי בית / המשך", [
        ("ה' - submit_job", check_e1),
        ("ה' - worker_loop", check_e2),
        ("ה' - get_status", check_e3),
        ("ה' - status_counts", check_e4),
        ("ו' - parse_dockerfile", check_f1),
        ("ו' - כללי lint ל-Dockerfile", check_f2),
        ("ו' - compose_services", check_f3),
        ("ו' - k8s_replicas", check_f4),
        ("ז' - get_rate (אופציונלי)", check_g),
    ]),
]
for title, checks in groups:
    print(f"--- {title} ---")
    for name, check in checks:
        try:
            check()
            print(f"✓ {name}")
        except NotImplementedError:
            print(f"✗ {name}  ← עדיין לא מומש")
        except AssertionError as e:
            print(f"✗ {name}  ← {e}")
        except Exception as e:
            print(f"✗ {name}  ← שגיאה בקוד ({type(e).__name__}: {e})")
    print()